# 03_V1_DigitalTwin_ScenarioDesign

## Physics-Informed Predictive Battery Digital Twin — V1 Specification

**Purpose:** Authoritative blueprint defining V1 multi-physics inputs, outputs, operational envelope, 100-cell virtual pack (Cell 37 target), risk engine contracts, S1–S6 scenarios, and counterfactual interventions.

**Strict Scientific Boundary:**
- No PINN model training or weight adjustments occur in this notebook.
- No frontend, UI widgets, or 3D renderings are constructed.
- No fabricated results or unverified predictions are stored.
- Downstream handoff: `04_V1_PyBaMM_DataGenerator.ipynb`.

In [1]:
# Cell 2: Standard Library & Scientific Imports
import os
import sys
import json
import math
import copy
import hashlib
import datetime
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

print("Cell 2: Standard scientific libraries loaded successfully.")

Cell 2: Standard scientific libraries loaded successfully.


In [2]:
# Cell 3: Runtime Verification (CPU-Only / Deterministic)
try:
    import torch
    gpu_active = torch.cuda.is_available()
except ImportError:
    gpu_active = False

print(f"Cell 3: Execution environment verified. GPU active: {gpu_active} (Scenario design is architecture/hardware agnostic).")

Cell 3: Execution environment verified. GPU active: False (Scenario design is architecture/hardware agnostic).


In [3]:
# Cell 4: Dynamic Google Drive Discovery & Strict Project Root Resolution
import os
from pathlib import Path
from google.colab import drive

candidate_roots = [
    Path("/content/drive/MyDrive"),
    Path("/content/gdrive_retry/MyDrive"),
    Path("/content/gdrive/MyDrive"),
    Path("/content/gdrive_pinn/MyDrive"),
]

existing_roots = [p for p in candidate_roots if p.exists()]

if not existing_roots:
    drive.mount("/content/drive", force_remount=False)
    existing_roots = [p for p in candidate_roots if p.exists()]

project_candidates = [
    root / "PINN_EV_Battery"
    for root in existing_roots
    if (root / "PINN_EV_Battery").is_dir()
]

if not project_candidates:
    raise FileNotFoundError(
        "DO_NOT_PROCEED: PINN_EV_Battery project root was not found in any mounted Drive location. "
        "No fallback project or directories will be created."
    )

PROJECT_ROOT = project_candidates[0]
DATA_DIR = PROJECT_ROOT / "data"
SCENARIO_DIR = PROJECT_ROOT / "scenario_design"
RESULTS_DIR = PROJECT_ROOT / "results"

print("Cell 4: PASS — Genuine project root verified.")
print(f"  PROJECT_ROOT: {PROJECT_ROOT}")
print(f"  DATA_DIR    : {DATA_DIR}")
print(f"  SCENARIO_DIR: {SCENARIO_DIR}")

Cell 4: PASS — Genuine project root verified.
  PROJECT_ROOT: /content/drive/MyDrive/PINN_EV_Battery
  DATA_DIR    : /content/drive/MyDrive/PINN_EV_Battery/data
  SCENARIO_DIR: /content/drive/MyDrive/PINN_EV_Battery/scenario_design


In [4]:
# Cell 5: Authoritative Source Discovery & Verified Byte-for-Byte Timestamped Backup
import glob
import hashlib
import datetime
from pathlib import Path

# A real source notebook must already exist. Never fabricate one.
search_roots = [
    Path("/content/drive/MyDrive/Colab Notebooks"),
    Path("/content/gdrive_retry/MyDrive/Colab Notebooks"),
    Path("/content/gdrive/MyDrive/Colab Notebooks"),
    Path("/content/gdrive_pinn/MyDrive/Colab Notebooks"),
    Path(PROJECT_ROOT),
]

nb_candidates = []
for root in search_roots:
    if root.exists():
        nb_candidates.extend(root.rglob("*03_V1_DigitalTwin_ScenarioDesign*.ipynb"))

nb_candidates = [
    p for p in nb_candidates
    if "backup" not in p.name.lower()
    and p.is_file()
    and p.stat().st_size > 100
]

# Validate candidate notebooks as real Jupyter notebooks.
valid_sources = []
for p in nb_candidates:
    try:
        with open(p, "r", encoding="utf-8") as fh:
            candidate_json = json.load(fh)
        if isinstance(candidate_json, dict) and isinstance(candidate_json.get("cells"), list) and len(candidate_json["cells"]) > 0:
            valid_sources.append(p)
    except Exception:
        pass

if not valid_sources:
    raise FileNotFoundError(
        "DO_NOT_PROCEED: authoritative Notebook 03 source was not discovered. "
        "Save the active notebook to Google Drive (preferably MyDrive/Colab Notebooks/) and rerun Cell 5. "
        "No fallback notebook will be created."
    )

source_nb = max(valid_sources, key=lambda p: p.stat().st_mtime)

src_bytes = source_nb.read_bytes()
src_hash = hashlib.sha256(src_bytes).hexdigest()

SCENARIO_DIR.mkdir(parents=True, exist_ok=True)

ts = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
BACKUP_PATH = SCENARIO_DIR / f"03_notebook_source_{ts}.backup.ipynb"
BACKUP_PATH.write_bytes(src_bytes)

bak_bytes = BACKUP_PATH.read_bytes()
bak_hash = hashlib.sha256(bak_bytes).hexdigest()

assert len(src_bytes) == len(bak_bytes), "BACKUP INTEGRITY FAIL: byte length mismatch."
assert src_hash == bak_hash, "BACKUP INTEGRITY FAIL: SHA-256 mismatch."

V0_DATASET_PATH = DATA_DIR / "dataset.csv"
V0_METADATA_PATH = DATA_DIR / "metadata.json"

print("Cell 5: PASS — real source notebook discovered and backed up byte-for-byte.")
print(f"  Source : {source_nb} ({len(src_bytes):,} bytes)")
print(f"  Backup : {BACKUP_PATH}")
print(f"  SHA256 : {src_hash}")


Cell 5: PASS — real source notebook discovered and backed up byte-for-byte.
  Source : /content/drive/MyDrive/Colab Notebooks/03_V1_DigitalTwin_ScenarioDesign_FINAL_DYNAMIC_LOCALIZATION (1).ipynb (225,865 bytes)
  Backup : /content/drive/MyDrive/PINN_EV_Battery/scenario_design/03_notebook_source_20260926_152606.backup.ipynb
  SHA256 : a4b080beeb595b05ec1509af54131afef7511266d8dd2d14febcc9a3ac86a775


In [5]:
# Cell 6 & 7: Strict V0 Artifact, Exact Schema, Condition/Split and Metadata Consistency

assert V0_DATASET_PATH.is_file(), (
    f"DO_NOT_PROCEED: Missing V0 dataset: {V0_DATASET_PATH}"
)
assert V0_METADATA_PATH.is_file(), (
    f"DO_NOT_PROCEED: Missing V0 metadata: {V0_METADATA_PATH}"
)

EXPECTED_V0_COLUMNS = [
    "c_rate",
    "T_amb_C",
    "r_m",
    "t_s",
    "r_norm",
    "t_norm",
    "c_mol_m3",
    "c_norm",
    "T_K",
    "T_rise_K",
    "V_V",
    "q_W_m3",
    "sigma_r_MPa",
    "sigma_t_MPa",
    "split",
    "cond_id"
]

V0_DATASET_SHA256 = hashlib.sha256(
    V0_DATASET_PATH.read_bytes()
).hexdigest()

V0_METADATA_SHA256 = hashlib.sha256(
    V0_METADATA_PATH.read_bytes()
).hexdigest()

with open(V0_METADATA_PATH, "r", encoding="utf-8") as f:
    V0_METADATA = json.load(f)

assert V0_METADATA.get("mock_pinn") is not True, (
    "DO_NOT_PROCEED: V0 metadata contains mock_pinn=True."
)

v0_df = pd.read_csv(V0_DATASET_PATH)

assert v0_df.shape == (43200, 16), (
    f"V0 shape mismatch: {v0_df.shape}; expected (43200, 16)."
)

assert v0_df.columns.tolist() == EXPECTED_V0_COLUMNS, (
    "Exact V0 column schema mismatch.\n"
    f"Expected: {EXPECTED_V0_COLUMNS}\n"
    f"Found: {v0_df.columns.tolist()}"
)

assert not v0_df.isna().any().any(), "V0 contains NaN values."

numeric_v0 = v0_df.select_dtypes(include=[np.number])
assert np.isfinite(numeric_v0.to_numpy()).all(), (
    "V0 contains non-finite numeric values."
)

assert set(v0_df["split"].unique()) == {"train", "val", "test"}, (
    f"Unexpected split labels: {sorted(v0_df['split'].unique().tolist())}"
)

condition_split_counts = (
    v0_df.groupby("cond_id")["split"].nunique()
)

assert (condition_split_counts == 1).all(), (
    "At least one condition appears in more than one split."
)

unique_conds = sorted(v0_df["cond_id"].unique().tolist())
assert len(unique_conds) == 20, (
    f"Expected 20 unique conditions, found {len(unique_conds)}."
)

split_counts = (
    v0_df.groupby("split")["cond_id"].nunique().to_dict()
)

assert split_counts == {
    "train": 12,
    "val": 6,
    "test": 2
}, f"Expected 12/6/2 condition split, found {split_counts}"

# Metadata structure must be present, not merely assumed.
required_metadata_keys = {"conditions", "n_rows"}
missing_meta = required_metadata_keys - set(V0_METADATA)
assert not missing_meta, (
    f"V0 metadata missing required keys: {sorted(missing_meta)}"
)

assert V0_METADATA["n_rows"] == 43200, (
    f"Metadata n_rows mismatch: {V0_METADATA.get('n_rows')}"
)

meta_conditions = V0_METADATA["conditions"]
assert isinstance(meta_conditions, list)
assert len(meta_conditions) == 20, (
    f"Metadata contains {len(meta_conditions)} conditions; expected 20."
)

def _metadata_condition_id(item):
    if not isinstance(item, dict):
        return None
    for key in ("cond_id", "condition_id", "id"):
        if key in item:
            return item[key]
    return None

metadata_cond_ids = [_metadata_condition_id(x) for x in meta_conditions]
assert all(x is not None for x in metadata_cond_ids), (
    "Could not resolve a condition ID for every metadata condition."
)

metadata_cond_ids = sorted(metadata_cond_ids)

assert metadata_cond_ids == unique_conds, (
    "Dataset condition IDs and metadata condition IDs do not match."
)

V0_LINEAGE = {
    "status": "VERIFIED",
    "dataset_path": str(V0_DATASET_PATH),
    "metadata_path": str(V0_METADATA_PATH),
    "dataset_sha256": V0_DATASET_SHA256,
    "metadata_sha256": V0_METADATA_SHA256,
    "shape": [43200, 16],
    "columns": EXPECTED_V0_COLUMNS,
    "condition_count": 20,
    "split_condition_counts": split_counts,
    "dataset_condition_ids": unique_conds,
    "metadata_condition_ids": metadata_cond_ids,
    "mock_pinn": bool(V0_METADATA.get("mock_pinn", False)),
    "metadata_keys_checked": sorted(required_metadata_keys),
}

V0_ARTIFACT_HASHES = {
    "dataset.csv": V0_DATASET_SHA256,
    "metadata.json": V0_METADATA_SHA256,
}

print("Cell 6 & 7: PASS")
print("Exact V0 schema:", len(EXPECTED_V0_COLUMNS), "columns")
print("Rows:", len(v0_df))
print("Conditions:", V0_LINEAGE["condition_count"])
print("Split:", V0_LINEAGE["split_condition_counts"])
print("Dataset/metadata condition IDs: MATCH")
print("V0 lineage:", V0_LINEAGE["status"])


Cell 6 & 7: PASS
Exact V0 schema: 16 columns
Rows: 43200
Conditions: 20
Split: {'test': 2, 'train': 12, 'val': 6}
Dataset/metadata condition IDs: MATCH
V0 lineage: VERIFIED


In [6]:
# Cell 8: Explicit V0 Capability Boundary Definition
V0_CAPABILITY_BOUNDARY = {
    "supported_inputs": ["r_norm", "t_norm", "c_rate", "t_amb"],
    "supported_outputs": ["concentration", "temperature", "radial_stress", "tangential_stress"],
    "regime": "constant_current_discharge_only",
    "cell_level": "single_cell_continuum",
    "charging_supported": False,
    "degradation_supported": False,
    "pack_network_supported": False,
    "intervention_supported": False,
    "risk_engine_supported": False
}

print("Cell 8: V0 Boundary explicitly documented. V1 capabilities will NOT be back-attributed to V0.")

Cell 8: V0 Boundary explicitly documented. V1 capabilities will NOT be back-attributed to V0.


In [7]:
# Cell 9: 100-Cell Pack with Dynamic Whole-Pack Health/Localization

PACK_CONFIG = {
    "pack_id": "VP_V1_CONFIG_01",
    "series_cells": 100,
    "parallel_strings": {
        "value": None,
        "status": "UNRESOLVED",
        "provenance": None
    },
    "monitored_cells": list(range(1, 101)),
    "cell_identifier_format": "CELL_{:03d}",

    # Cell 37 is ONLY a controlled benchmark experiment.
    "reference_experiment_cell_id": 37,

    "fault_localization": {
        "mode": "DYNAMIC_WHOLE_PACK",
        "candidate_cells": list(range(1, 101)),
        "supports_single_cell_events": True,
        "supports_multi_cell_events": True,
        "selection_basis": [
            "cell_voltage",
            "cell_current",
            "cell_temperature",
            "cell_soc",
            "capacity_state",
            "resistance_state",
            "degradation_state",
            "operating_history",
            "forecast_trajectories",
            "cross_cell_deviation"
        ],
        "decision_source": "DOWNSTREAM_VALIDATED_CELL_LEVEL_DATA",
        "random_fault_assignment": False
    },

    "nominal_voltage_v": {
        "value": None,
        "status": "UNRESOLVED",
        "provenance": None
    },
    "nominal_capacity_ah": {
        "value": None,
        "status": "UNRESOLVED",
        "provenance": None
    },
    "software_only": True
}

assert PACK_CONFIG["series_cells"] == 100
assert PACK_CONFIG["monitored_cells"] == list(range(1, 101))
assert PACK_CONFIG["reference_experiment_cell_id"] == 37
assert PACK_CONFIG["fault_localization"]["candidate_cells"] == list(range(1, 101))
assert PACK_CONFIG["fault_localization"]["random_fault_assignment"] is False

print("Cell 9: PASS — all 100 cells are monitored for dynamic health/fault localization.")
print("  Cell 37 is a controlled benchmark only; it is not a permanent fault location.")


Cell 9: PASS — all 100 cells are monitored for dynamic health/fault localization.
  Cell 37 is a controlled benchmark only; it is not a permanent fault location.


In [8]:
# Cell 10: Comprehensive V1 Input Specification Schema
INPUT_SPEC = [
    {
        "field": "cell_id",
        "type": "int",
        "unit": "index",
        "physical_meaning": "Cell position in pack sequence",
        "source": "BMS/pack_map",
        "scope": "cell",
        "valid_range": [1, 100],
        "availability": "measured",
        "classification": "measured"
    },
    {
        "field": "time_s",
        "type": "float",
        "unit": "s",
        "physical_meaning": "Elapsed cycle timestamp",
        "source": "simulator/clock",
        "scope": "system",
        "valid_range": [0.0, 7200.0],
        "availability": "measured",
        "classification": "measured"
    },
    {
        "field": "operating_state",
        "type": "string",
        "unit": "categorical",
        "physical_meaning": "Operating mode (CHARGE, DISCHARGE, REST)",
        "source": "BMS/controller",
        "scope": "pack",
        "valid_range": ["CHARGE", "DISCHARGE", "REST"],
        "availability": "control",
        "classification": "controlled"
    },
    {
        "field": "pack_voltage_v",
        "type": "float",
        "unit": "V",
        "physical_meaning": "Aggregate terminal voltage",
        "source": "BMS/sensor",
        "scope": "pack",
        "valid_range": [250.0, 430.0],
        "availability": "measured",
        "classification": "measured"
    },
    {
        "field": "pack_current_a",
        "type": "float",
        "unit": "A",
        "physical_meaning": "Aggregate pack current",
        "source": "BMS/sensor",
        "scope": "pack",
        "valid_range": [-300.0, 300.0],
        "availability": "measured",
        "classification": "measured"
    },
    {
        "field": "charging_state",
        "type": "bool",
        "unit": "boolean",
        "physical_meaning": "Charger actively connected and energizing",
        "source": "EVSE/charger",
        "scope": "system",
        "valid_range": [False, True],
        "availability": "control",
        "classification": "controlled"
    },
    {
        "field": "charging_rate_c",
        "type": "float",
        "unit": "C",
        "physical_meaning": "Applied normalized charging current",
        "source": "BMS/control",
        "scope": "system",
        "valid_range": [0.1, 3.5],
        "availability": "control",
        "classification": "controlled"
    },
    {
        "field": "ambient_temp_c",
        "type": "float",
        "unit": "degC",
        "physical_meaning": "Ambient environmental temperature",
        "source": "environmental_sensor",
        "scope": "system",
        "valid_range": [-20.0, 55.0],
        "availability": "measured",
        "classification": "measured"
    },
    {
        "field": "surface_temperature_c",
        "type": "float",
        "unit": "degC",
        "physical_meaning": "Cell outer casing temperature",
        "source": "BMS/thermistor",
        "scope": "cell",
        "valid_range": [-20.0, 80.0],
        "availability": "measured",
        "classification": "measured"
    },
    {
        "field": "current_soc",
        "type": "float",
        "unit": "fraction",
        "physical_meaning": "Instantaneous cell state of charge",
        "source": "BMS/estimator",
        "scope": "cell",
        "valid_range": [0.0, 1.0],
        "availability": "estimated",
        "classification": "estimated"
    },
    {
        "field": "initial_soc",
        "type": "float",
        "unit": "fraction",
        "physical_meaning": "State of charge at cycle onset",
        "source": "BMS/history",
        "scope": "cell",
        "valid_range": [0.0, 1.0],
        "availability": "measured",
        "classification": "measured"
    },
    {
        "field": "cycle_number",
        "type": "int",
        "unit": "cycles",
        "physical_meaning": "Cumulative equivalent full cycles",
        "source": "BMS/memory",
        "scope": "cell",
        "valid_range": [0, 5000],
        "availability": "history",
        "classification": "measured"
    },
    {
        "field": "operating_history",
        "type": "list[float]",
        "unit": "history_series",
        "physical_meaning": "Recent rolling load sequence",
        "source": "BMS/buffer",
        "scope": "cell",
        "valid_range": {"min_length": 10, "max_length": 120, "element_type": "float"},
        "availability": "historical",
        "classification": "measured"
    },
    {
        "field": "history_window_steps",
        "type": "int",
        "unit": "steps",
        "physical_meaning": "Window size for historical trajectory ingestion",
        "source": "digital_twin/config",
        "scope": "model",
        "valid_range": [10, 120],
        "availability": "parameter",
        "classification": "controlled"
    },
    {
        "field": "capacity_state",
        "type": "float",
        "unit": "fraction",
        "physical_meaning": "Relative retained cell capacity",
        "source": "digital_twin/estimator",
        "scope": "cell",
        "valid_range": [0.5, 1.0],
        "availability": "estimated",
        "classification": "estimated"
    },
    {
        "field": "resistance_state",
        "type": "float",
        "unit": "multiplier",
        "physical_meaning": "Internal resistance multiplier relative to BOL",
        "source": "digital_twin/estimator",
        "scope": "cell",
        "valid_range": [1.0, 3.0],
        "availability": "estimated",
        "classification": "estimated"
    },
    {
        "field": "degradation_state",
        "type": "string",
        "unit": "categorical",
        "physical_meaning": "Cell physical condition flag",
        "source": "digital_twin/classifier",
        "scope": "cell",
        "valid_range": ["healthy", "degraded"],
        "availability": "derived",
        "classification": "estimated"
    },
    {
        "field": "cooling_flow_state",
        "type": "float",
        "unit": "fraction",
        "physical_meaning": "Cooling pump fractional flow rate",
        "source": "thermal_management",
        "scope": "pack",
        "valid_range": [0.0, 1.0],
        "availability": "control",
        "classification": "controlled"
    },
    {
        "field": "spatial_r_norm",
        "type": "float",
        "unit": "dimensionless",
        "physical_meaning": "Normalized particle radial coordinate",
        "source": "continuum_mesh",
        "scope": "internal",
        "valid_range": [0.0, 1.0],
        "availability": "simulation",
        "classification": "latent"
    }
]

# Validation of required classification domain
ALLOWED_CLASSIFICATIONS = {"measured", "controlled", "estimated", "latent", "training_label"}
for item in INPUT_SPEC:
    assert item["classification"] in ALLOWED_CLASSIFICATIONS, (
        f"Invalid classification '{item['classification']}' in field '{item['field']}'"
    )

print(f"Cell 10: PASS — Complete V1 Input Specification defined with {len(INPUT_SPEC)} fields.")
print("  All field classifications strictly aligned to allowed set.")

Cell 10: PASS — Complete V1 Input Specification defined with 19 fields.
  All field classifications strictly aligned to allowed set.


In [9]:
# Cell 11: Complete V1 Output Specification with Whole-Pack Cell Health/Localization

OUTPUT_SPEC = [
    {"field": "solid_particle_concentration", "unit": "mol/m3", "category": "electrochemical", "domain": "solid_particle", "meaning": "Lithium concentration field within the solid active-material particle, indexed over the particle radial coordinate.", "status": "REQUIRED"},
    {"field": "electrolyte_concentration", "unit": "mol/m3", "category": "electrochemical", "domain": "electrolyte", "meaning": "Lithium-ion concentration in electrolyte regions; never substituted for solid-particle concentration.", "status": "REQUIRED"},
    {"field": "cell_temperature", "unit": "degC", "category": "thermal", "domain": "cell", "meaning": "Predicted cell temperature at the declared spatial/aggregation level.", "status": "REQUIRED"},
    {"field": "radial_stress", "unit": "MPa", "category": "mechanical", "domain": "particle_bulk", "coordinate": "radial", "meaning": "Radial stress component as defined by the downstream constitutive/mechanical model.", "status": "REQUIRED"},
    {"field": "tangential_stress", "unit": "MPa", "category": "mechanical", "domain": "particle_bulk", "coordinate": "tangential/hoop", "meaning": "Tangential/hoop stress component as defined by the downstream constitutive/mechanical model.", "status": "REQUIRED"},
    {"field": "capacity_retention_state", "unit": "fraction", "category": "health", "domain": "cell", "meaning": "Retained capacity fraction relative to beginning-of-life/reference capacity.", "status": "REQUIRED"},
    {"field": "resistance_multiplier_state", "unit": "multiplier", "category": "health", "domain": "cell", "meaning": "Internal resistance relative to the declared beginning-of-life reference.", "status": "REQUIRED"},
    {"field": "soh_definition", "unit": "string", "category": "health", "domain": "definition", "meaning": "SOH definition must state the quantity, reference state, and calculation convention actually used.", "status": "REQUIRED"},

    {"field": "future_solid_particle_concentration_trajectory", "unit": "mol/m3_array", "category": "forecast", "domain": "solid_particle", "meaning": "Forecast trajectory of solid-particle lithium concentration.", "status": "REQUIRED"},
    {"field": "future_electrolyte_concentration_trajectory", "unit": "mol/m3_array", "category": "forecast", "domain": "electrolyte", "meaning": "Forecast trajectory of electrolyte concentration.", "status": "REQUIRED"},
    {"field": "future_temperature_trajectory", "unit": "degC_array", "category": "forecast", "domain": "cell", "meaning": "Forecast cell temperature trajectory.", "status": "REQUIRED"},
    {"field": "future_radial_stress_trajectory", "unit": "MPa_array", "category": "forecast", "domain": "particle_bulk", "coordinate": "radial", "meaning": "Forecast radial stress trajectory.", "status": "REQUIRED"},
    {"field": "future_tangential_stress_trajectory", "unit": "MPa_array", "category": "forecast", "domain": "particle_bulk", "coordinate": "tangential/hoop", "meaning": "Forecast tangential stress trajectory.", "status": "REQUIRED"},
    {"field": "future_soh_trajectory", "unit": "fraction_array", "category": "forecast", "domain": "cell", "meaning": "Forecast health/SOH trajectory.", "status": "REQUIRED"},

    # Whole-pack localization outputs. These are schema contracts only in Notebook 03.
    {"field": "per_cell_health_state", "unit": "categorical_array", "category": "localization", "domain": "pack_cells", "meaning": "Health state for every Cell ID 1–100 after downstream validated cell-level evaluation.", "status": "REQUIRED", "allowed_states": ["HEALTHY", "AFFECTED", "DEFECTIVE", "POTENTIALLY_AFFECTED", "UNASSESSED"]},
    {"field": "per_cell_health_score", "unit": "dimensionless_array", "category": "localization", "domain": "pack_cells", "meaning": "Per-cell health score produced by the downstream validated health model; scale must be defined before operational use.", "status": "REQUIRED"},
    {"field": "per_cell_anomaly_score", "unit": "dimensionless_array", "category": "localization", "domain": "pack_cells", "meaning": "Per-cell deviation/anomaly score relative to a declared healthy reference distribution or model envelope.", "status": "REQUIRED"},
    {"field": "defective_cells", "unit": "cell_id_list", "category": "localization", "domain": "pack_cells", "meaning": "Cells whose validated defect criteria are crossed; may contain zero, one, or multiple cell IDs.", "status": "REQUIRED"},
    {"field": "affected_cells", "unit": "cell_id_list", "category": "localization", "domain": "pack_cells", "meaning": "Cells with validated current abnormal/impact evidence but without the declared defective-cell criterion being crossed.", "status": "REQUIRED"},
    {"field": "potentially_affected_cells", "unit": "cell_id_list", "category": "localization", "domain": "pack_cells", "meaning": "Cells whose forecast or cross-cell evidence indicates elevated future risk within the declared horizon.", "status": "REQUIRED"},
    {"field": "healthy_cells", "unit": "cell_id_list", "category": "localization", "domain": "pack_cells", "meaning": "Cells with no validated current or forecast abnormality under the declared evaluation criteria.", "status": "REQUIRED"},
    {"field": "cell_localization_confidence", "unit": "probability_or_confidence", "category": "localization", "domain": "pack_cells", "meaning": "Confidence associated with each localized cell classification; method must be validated downstream.", "status": "REQUIRED"},
    {"field": "localization_evidence", "unit": "structured_record_array", "category": "localization", "domain": "pack_cells", "meaning": "Per-cell evidence linking observed/modelled voltage, thermal, SOC, health, history, stress and concentration deviations to the classification.", "status": "REQUIRED"},
    {"field": "localization_trigger", "unit": "categorical", "category": "localization", "domain": "pack", "meaning": "Declared trigger for whole-pack evaluation, such as a validated pack-level electrical, thermal or predictive anomaly.", "status": "REQUIRED"},

    {"field": "safety_envelope_margin", "unit": "dimensionless", "category": "risk", "domain": "evaluation", "meaning": "Domain-specific normalized distance to a validated boundary; incompatible raw units are not combined directly.", "status": "REQUIRED"},
    {"field": "time_to_threshold_s", "unit": "s", "category": "risk", "domain": "temporal", "meaning": "Forecast time until the earliest applicable validated constraint crossing.", "status": "REQUIRED"},
    {"field": "rul_projection_cycles", "unit": "cycles", "category": "forecast", "domain": "lifetime", "meaning": "Projected remaining useful life under an explicitly stated future-use assumption.", "status": "REQUIRED"},
    {"field": "rul_eol_criterion", "unit": "string", "category": "forecast", "domain": "lifetime", "meaning": "Explicit end-of-life criterion used for RUL.", "status": "REQUIRED"},
    {"field": "rul_future_use_assumption", "unit": "string", "category": "forecast", "domain": "lifetime", "meaning": "Explicit future operating/use assumption used by the RUL projection.", "status": "REQUIRED"},
    {"field": "risk_level", "unit": "categorical", "category": "decision", "domain": "logic", "meaning": "Pack/cell risk tier after validated boundaries are available.", "status": "REQUIRED"},
    {"field": "recommended_action", "unit": "categorical", "category": "decision", "domain": "control", "meaning": "Action generated only after validated risk logic is available.", "status": "REQUIRED"}
]

print(f"Cell 11: PASS — {len(OUTPUT_SPEC)} outputs defined, including 100-cell health/localization outputs.")


Cell 11: PASS — 31 outputs defined, including 100-cell health/localization outputs.


In [10]:
# Cell 12: Cell-Agnostic Degradation Contract + Cell 37 Benchmark

DEGRADATION_CONTRACT = {
    "mode": "CELL_AGNOSTIC",
    "candidate_cells": list(range(1, 101)),
    "supports_single_cell": True,
    "supports_multi_cell": True,
    "random_fault_assignment": False,
    "selection_basis": "DOWNSTREAM_PHYSICAL_STATE_AND_VALIDATED_LOCALIZATION",
    "benchmark": {
        "cell_id": 37,
        "purpose": "controlled_ground_truth_localization_experiment"
    },
    "parameters": {
        "resistance_multiplier": {
            "provisional_value": 1.8,
            "status": "PROVISIONAL",
            "source": None
        },
        "capacity_retention": {
            "provisional_value": 0.82,
            "status": "PROVISIONAL",
            "source": None
        },
        "thermal_conductivity_factor": {
            "provisional_value": 0.85,
            "status": "PROVISIONAL",
            "source": None
        }
    },
    "required_downstream_mapping":
        "Map degradation to an actual PyBaMM chemistry/aging representation before physical claims are made."
}

# Backward-compatible alias used by downstream existing cells.
CELL37_DEGRADATION_CONTRACT = {
    "target_cell_id": 37,
    "healthy_reference_cells": [i for i in range(1, 101) if i != 37],
    "degradation_mode": {
        "label": "controlled_benchmark_within_cell_agnostic_framework",
        "status": "PROVISIONAL"
    },
    "parameters": DEGRADATION_CONTRACT["parameters"]
}

assert DEGRADATION_CONTRACT["candidate_cells"] == list(range(1, 101))
assert DEGRADATION_CONTRACT["benchmark"]["cell_id"] == 37
assert DEGRADATION_CONTRACT["random_fault_assignment"] is False

print("Cell 12: PASS — degradation is cell-agnostic; Cell 37 is benchmark only.")


Cell 12: PASS — degradation is cell-agnostic; Cell 37 is benchmark only.


In [11]:
# Cell 13: Scenarios S1 to S3 — Whole-Pack Evaluation

s1 = {
    "scenario_id": "S1",
    "name": "Normal Operation",
    "description": "Healthy 100-cell pack under nominal declared conditions; all cells are evaluated.",
    "target_cell": None,
    "affected_cells_mode": "NONE_EXPECTED",
    "matched_healthy_reference": "All 100 cells nominal",
    "changes": {"degradation_state": "healthy"},
    "inputs": {"ambient_temp_c": 25.0, "target_charge_rate_c": 1.0, "initial_soc": 0.2},
    "prediction_intent": "Evaluate health/state of every cell and establish the healthy reference distribution."
}

s2 = {
    "scenario_id": "S2",
    "name": "High Charging Stress",
    "description": "Charging rate increased while the whole 100-cell pack is scanned for cell-level divergence.",
    "target_cell": None,
    "affected_cells_mode": "AUTO_DETECT",
    "matched_healthy_reference": "S1 with identical ambient temperature and healthy pack state",
    "changes": {"degradation_state": "healthy"},
    "inputs": {"ambient_temp_c": 25.0, "target_charge_rate_c": 2.5, "initial_soc": 0.2},
    "prediction_intent": "Evaluate every cell for electrical/thermal/state deviation rather than assuming a predetermined fault location."
}

s3 = {
    "scenario_id": "S3",
    "name": "High Ambient Temperature",
    "description": "Ambient temperature elevated while keeping charging control matched to S1; every cell is evaluated.",
    "target_cell": None,
    "affected_cells_mode": "AUTO_DETECT",
    "matched_healthy_reference": "S1 with identical charge rate and healthy pack state",
    "changes": {"degradation_state": "healthy"},
    "inputs": {"ambient_temp_c": 45.0, "target_charge_rate_c": 1.0, "initial_soc": 0.2},
    "prediction_intent": "Evaluate whether any subset of cells develops abnormal thermal/electrical state under elevated ambient conditions."
}

print("Cell 13: PASS — S1-S3 scan all 100 cells; no cell is preselected as defective.")


Cell 13: PASS — S1-S3 scan all 100 cells; no cell is preselected as defective.


In [12]:
# Cell 14: Scenarios S4 to S6 — Dynamic Cell Localization

s4 = {
    "scenario_id": "S4",
    "name": "Localized Degradation Detection",
    "description": "Detect degradation anywhere in the 100-cell pack. Cell 37 is retained only as the controlled benchmark injection case.",
    "target_cell": None,
    "affected_cells_mode": "DECLARED_PER_RUN",
    "benchmark_experiment_cell": 37,
    "matched_healthy_reference": "Same 100-cell operating condition without injected/known degradation",
    "changes": {
        "degradation_state": "cell_specific_and_declared_per_run"
    },
    "inputs": {"ambient_temp_c": 25.0, "target_charge_rate_c": 1.0, "initial_soc": 0.2},
    "prediction_intent": "Identify which cell or cells deviate from the validated healthy reference without assuming Cell 37."
}

s5 = {
    "scenario_id": "S5",
    "name": "Combined High-Risk Scenario",
    "description": "Combine cell-specific degradation with elevated charging and ambient temperature; scan all 100 cells for direct and propagated effects.",
    "target_cell": None,
    "affected_cells_mode": "INHERIT_OR_DECLARE_PER_RUN",
    "benchmark_experiment_cell": 37,
    "matched_healthy_reference": "Matched healthy 100-cell pack under the same operating controls",
    "changes": copy.deepcopy(s4["changes"]),
    "inputs": {"ambient_temp_c": 45.0, "target_charge_rate_c": 2.0, "initial_soc": 0.2},
    "prediction_intent": "Localize defective cells, currently affected cells, and potentially affected cells from whole-pack multi-physics evidence."
}

s6 = {
    "scenario_id": "S6",
    "name": "Intervention / What-If Recovery",
    "description": "Inherit S5 cell-specific state and evaluate candidate control changes while scanning all cells again.",
    "target_cell": None,
    "affected_cells_mode": "INHERIT_FROM_S5",
    "matched_healthy_reference": "S5 without intervention",
    "changes": copy.deepcopy(s5["changes"]),
    "inputs": {"ambient_temp_c": 45.0, "target_charge_rate_c": 0.4, "initial_soc": 0.2},
    "prediction_intent": "Determine how intervention changes per-cell trajectories and whether risk/localization changes over the full forecast horizon."
}

SCENARIOS = [s1, s2, s3, s4, s5, s6]

print("Cell 14: PASS — S4-S6 use dynamic cell localization; Cell 37 is benchmark only.")


Cell 14: PASS — S4-S6 use dynamic cell localization; Cell 37 is benchmark only.


In [13]:
# Cell 15: S6 Counterfactual Sweep with Cell-State Invariance

S6_INTERVENTION_SWEEP = {
    "base_scenario": "S5",
    "affected_cells_source": "INHERIT_FROM_S5",
    "candidate_cells_scanned_each_run": list(range(1, 101)),
    "tested_candidate_c_rates": [2.0, 1.5, 1.0, 0.8, 0.4],
    "unchanged_rate_comparator": 2.0,
    "changed_rate_candidates": [1.5, 1.0, 0.8, 0.4],
    "winner_preselected": None,
    "selection_rule": (
        "After validated constraints are available, choose the highest tested C-rate "
        "satisfying all applicable validated constraints over the complete forecast horizon. "
        "Otherwise emit NO_FEASIBLE_TESTED_CANDIDATE."
    ),
    "invariance_assertion": (
        "Each S6 candidate must inherit the exact S5 per-cell degradation state, affected-cell set, "
        "ambient temperature, initial SOC, cooling state and other physical state. Only declared "
        "intervention controls may change."
    )
}

assert S6_INTERVENTION_SWEEP["candidate_cells_scanned_each_run"] == list(range(1, 101))
assert len(set(S6_INTERVENTION_SWEEP["tested_candidate_c_rates"])) == 5
assert S6_INTERVENTION_SWEEP["winner_preselected"] is None
assert any(c != 2.0 for c in S6_INTERVENTION_SWEEP["tested_candidate_c_rates"])

print("Cell 15: PASS — every S6 candidate rescans all 100 cells with unchanged cell-state inheritance.")


Cell 15: PASS — every S6 candidate rescans all 100 cells with unchanged cell-state inheritance.


In [14]:
# Cell 16: Complete Scenario Matrix with Whole-Pack Localization Contract

matrix_rows = []

for s_scen in SCENARIOS[:-1]:
    matrix_rows.append({
        "scenario_id": s_scen["scenario_id"],
        "sub_id": "BASE",
        "name": s_scen["name"],
        "target_cell": "DYNAMIC" if s_scen["target_cell"] is None else s_scen["target_cell"],
        "affected_cells_mode": s_scen.get("affected_cells_mode"),
        "benchmark_experiment_cell": s_scen.get("benchmark_experiment_cell"),
        "ambient_temp_c": s_scen["inputs"]["ambient_temp_c"],
        "charge_rate_c": s_scen["inputs"]["target_charge_rate_c"],
        "initial_soc": s_scen["inputs"]["initial_soc"],
        "degradation_state": s_scen["changes"]["degradation_state"],
        "cells_scanned": 100,
        "purpose": s_scen["prediction_intent"]
    })

s6_base = SCENARIOS[-1]
for c_cand in S6_INTERVENTION_SWEEP["tested_candidate_c_rates"]:
    matrix_rows.append({
        "scenario_id": "S6",
        "sub_id": f"CANDIDATE_{c_cand}C",
        "name": f"Intervention Sweep ({c_cand}C)",
        "target_cell": "DYNAMIC",
        "affected_cells_mode": "INHERIT_FROM_S5",
        "benchmark_experiment_cell": 37,
        "ambient_temp_c": s6_base["inputs"]["ambient_temp_c"],
        "charge_rate_c": c_cand,
        "initial_soc": s6_base["inputs"]["initial_soc"],
        "degradation_state": s6_base["changes"]["degradation_state"],
        "cells_scanned": 100,
        "purpose": f"Evaluate whole-pack per-cell trajectories and localization under {c_cand}C intervention"
    })

scenario_matrix_df = pd.DataFrame(matrix_rows)

assert len(scenario_matrix_df) == 10
assert scenario_matrix_df["cells_scanned"].eq(100).all()

print(f"Cell 16: PASS — {len(scenario_matrix_df)} cases; every case scans all 100 cells.")
display(scenario_matrix_df)


Cell 16: PASS — 10 cases; every case scans all 100 cells.


,scenario_id,sub_id,name,target_cell,affected_cells_mode,benchmark_experiment_cell,ambient_temp_c,charge_rate_c,initial_soc,degradation_state,cells_scanned,purpose
0,S1,BASE,Normal Operation,DYNAMIC,NONE_EXPECTED,NaN,25.0,1.0,0.2,healthy,100,Evaluate health/state of every cell and establ...
1,S2,BASE,High Charging Stress,DYNAMIC,AUTO_DETECT,NaN,25.0,2.5,0.2,healthy,100,Evaluate every cell for electrical/thermal/sta...
2,S3,BASE,High Ambient Temperature,DYNAMIC,AUTO_DETECT,NaN,45.0,1.0,0.2,healthy,100,Evaluate whether any subset of cells develops ...
3,S4,BASE,Localized Degradation Detection,DYNAMIC,DECLARED_PER_RUN,37.0,25.0,1.0,0.2,cell_specific_and_declared_per_run,100,Identify which cell or cells deviate from the ...
4,S5,BASE,Combined High-Risk Scenario,DYNAMIC,INHERIT_OR_DECLARE_PER_RUN,37.0,45.0,2.0,0.2,cell_specific_and_declared_per_run,100,"Localize defective cells, currently affected c..."
5,S6,CANDIDATE_2.0C,Intervention Sweep (2.0C),DYNAMIC,INHERIT_FROM_S5,37.0,45.0,2.0,0.2,cell_specific_and_declared_per_run,100,Evaluate whole-pack per-cell trajectories and ...
6,S6,CANDIDATE_1.5C,Intervention Sweep (1.5C),DYNAMIC,INHERIT_FROM_S5,37.0,45.0,1.5,0.2,cell_specific_and_declared_per_run,100,Evaluate whole-pack per-cell trajectories and ...
7,S6,CANDIDATE_1.0C,Intervention Sweep (1.0C),DYNAMIC,INHERIT_FROM_S5,37.0,45.0,1.0,0.2,cell_specific_and_declared_per_run,100,Evaluate whole-pack per-cell trajectories and ...
8,S6,CANDIDATE_0.8C,Intervention Sweep (0.8C),DYNAMIC,INHERIT_FROM_S5,37.0,45.0,0.8,0.2,cell_specific_and_declared_per_run,100,Evaluate whole-pack per-cell trajectories and ...
9,S6,CANDIDATE_0.4C,Intervention Sweep (0.4C),DYNAMIC,INHERIT_FROM_S5,37.0,45.0,0.4,0.2,cell_specific_and_declared_per_run,100,Evaluate whole-pack per-cell trajectories and ...


In [15]:
# Cell 17: Whole-Pack Dynamic Health & Fault-Localization Experiment

CELL_LOCALIZATION_EXPERIMENT = {

    "experiment_id":
        "EXP_PACK_WIDE_DYNAMIC_CELL_LOCALIZATION",

    "candidate_cells":
        list(range(1, 101)),

    "benchmark_cell":
        37,

    "benchmark_purpose":
        "Known-location controlled degradation case used only "
        "to validate localization against known ground truth.",

    "general_mode":
        "UNKNOWN_LOCATION_AT_INFERENCE",

    "operational_evaluation_mode":
        "EVENT_OR_TIME_STEP_TRIGGERED",

    "evaluation_triggers": [
        "pack_voltage_anomaly",
        "pack_current_anomaly",
        "cell_voltage_imbalance",
        "thermal_anomaly",
        "surface_temperature_anomaly",
        "soc_anomaly",
        "health_state_anomaly",
        "predicted_constraint_risk",
        "forecast_deviation",
        "operator_requested_scan"
    ],

    "scan_policy":
        "Whenever a declared evaluation trigger occurs, "
        "evaluate ALL 100 cells. Never restrict inference "
        "to Cell 37 or to a randomly selected cell.",

    "required_cell_level_evidence": [
        "cell_voltage",
        "cell_temperature",
        "cell_soc",
        "capacity_state",
        "resistance_state",
        "degradation_state",
        "operating_history",
        "forecast_trajectories",
        "cross_cell_deviation"
    ],

    "supported_outcomes": [
        "NO_ABNORMAL_CELL_DETECTED",
        "SINGLE_CELL_DEFECT",
        "MULTI_CELL_DEFECT",
        "CURRENTLY_AFFECTED_CELL_SET",
        "POTENTIALLY_AFFECTED_CELL_SET",
        "ALL_HEALTHY",
        "UNASSESSED"
    ],

    "classification_definitions": {

        "DEFECTIVE":
            "A validated defect criterion is crossed "
            "for the cell.",

        "AFFECTED":
            "Current validated evidence shows abnormal or "
            "degraded behavior, but the declared defective "
            "criterion is not crossed.",

        "POTENTIALLY_AFFECTED":
            "Forecast, neighboring-state, or cross-cell "
            "evidence indicates elevated future risk within "
            "the declared forecast horizon.",

        "HEALTHY":
            "No validated current or forecast abnormality "
            "is present under the declared evaluation criteria.",

        "UNASSESSED":
            "Required data, model, or threshold validation "
            "is unavailable."
    },

    "cell37_matched_healthy_case": {
        "cell_id": 37,
        "required": True,
        "control_matching": [
            "ambient_temperature",
            "charge_rate",
            "initial_soc",
            "cooling_state",
            "operating_history",
            "time_horizon"
        ],
        "only_intended_difference":
            "Cell 37 degradation state"
    },

    "no_hard_coded_fault_location":
        True,

    "no_random_fault_location":
        True,

    "localization_basis":
        "Cell-level physical/state evidence followed by "
        "validated localization logic."
}


assert (
    CELL_LOCALIZATION_EXPERIMENT["candidate_cells"]
    == list(range(1, 101))
)

assert (
    CELL_LOCALIZATION_EXPERIMENT["benchmark_cell"]
    == 37
)

assert (
    CELL_LOCALIZATION_EXPERIMENT["no_hard_coded_fault_location"]
    is True
)

assert (
    CELL_LOCALIZATION_EXPERIMENT["no_random_fault_location"]
    is True
)

assert (
    len(CELL_LOCALIZATION_EXPERIMENT["evaluation_triggers"])
    >= 5
)

print(
    "Cell 17: PASS — "
    "whole-pack trigger-driven localization defined for all 100 cells."
)
print(
    "Cell 37 = controlled benchmark only."
)

Cell 17: PASS — whole-pack localization experiment defined; Cell 37 remains benchmark only.


In [16]:
# Cell 18: Research Baseline Comparison Framework (B0–B3)
BASELINE_SPEC = [
    {
        "model_id": "B0",
        "name": "Conventional Threshold Monitoring",
        "physics_involvement": "None (fixed threshold alarms on macroscopic surface temperature/voltage)",
        "purpose": "Industry practical standard baseline"
    },
    {
        "model_id": "B1",
        "name": "Data-Only Neural Network",
        "physics_involvement": "None (pure empirical regression without conservation or stress equations)",
        "purpose": "Ablation baseline demonstrating necessity of physical loss terms"
    },
    {
        "model_id": "B2",
        "name": "Existing V0 PINN",
        "physics_involvement": "Multi-physics continuum (discharge CC SPMe only, no charging/aging)",
        "purpose": "Direct research ancestor baseline"
    },
    {
        "model_id": "B3",
        "name": "Proposed V1 Predictive PINN Digital Twin",
        "physics_involvement": "Full multi-physics + charging + localized degradation + forecast envelope",
        "purpose": "Proposed system"
    }
]

print("Cell 18: Research baselines B0–B3 verified.")

Cell 18: Research baselines B0–B3 verified.


In [17]:
# Cell 19: Complete Multi-Physics Risk Engine Specification & Status Separation
RISK_SPEC = {
    "engine_mode": "PROVISIONAL_SPECIFICATION",
    "operational_status": "UNASSESSED_UNTIL_VALIDATED",
    "principle": "Evaluate each physical domain against its own validated boundary and preserve units/domain semantics.",
    "domains": {},
    "legacy_draft_thresholds": {

    "status": "INACTIVE_DRAFT",

    "do_not_use_operationally": True,

    "historical_temperature_degC": [
        45.0,
        55.0,
        60.0,
        65.0
    ],

    "historical_stress_MPa": [
        1.0,
        2.0,
        3.5,
        5.0
    ],

    "mapping_status":
        "UNRESOLVED",

    "mapping_rule":
        "Historical draft values must not be assigned "
        "to radial stress, tangential stress, concentration, "
        "SOH, or any operational threshold without an "
        "authoritative source and chemistry/model applicability.",

    "operational_use":
        False
}
}

def _threshold_contract(quantity, domain, unit, direction):
    return {
        "quantity": quantity,
        "domain": domain,
        "value": None,
        "unit": unit,
        "source": None,
        "chemistry_applicability": "UNRESOLVED",
        "model_applicability": "UNRESOLVED",
        "rationale": "Threshold must be sourced/validated for the selected chemistry/model before operational use.",
        "validation_status": "UNRESOLVED",
        "boundary_rule": direction,
        "precedence": "Domain-specific validated boundary only; cross-domain aggregation requires explicit normalization."
    }

RISK_SPEC["domains"] = {
    "thermal": {
        "monitored_variable": "cell_temperature",
        "status": "UNRESOLVED",
        "thresholds": {
            "WARNING": _threshold_contract("cell_temperature", "cell", "degC", "upper"),
            "HIGH": _threshold_contract("cell_temperature", "cell", "degC", "upper"),
            "CRITICAL": _threshold_contract("cell_temperature", "cell", "degC", "upper")
        }
    },
    "mechanical_radial": {
        "monitored_variable": "radial_stress",
        "status": "UNRESOLVED",
        "thresholds": {
            "WARNING": _threshold_contract("radial_stress", "particle_bulk", "MPa", "upper"),
            "HIGH": _threshold_contract("radial_stress", "particle_bulk", "MPa", "upper"),
            "CRITICAL": _threshold_contract("radial_stress", "particle_bulk", "MPa", "upper")
        }
    },
    "mechanical_tangential": {
        "monitored_variable": "tangential_stress",
        "status": "UNRESOLVED",
        "thresholds": {
            "WARNING": _threshold_contract("tangential_stress", "particle_bulk", "MPa", "upper"),
            "HIGH": _threshold_contract("tangential_stress", "particle_bulk", "MPa", "upper"),
            "CRITICAL": _threshold_contract("tangential_stress", "particle_bulk", "MPa", "upper")
        }
    },
    "solid_concentration": {
        "monitored_variable": "solid_particle_concentration",
        "status": "UNRESOLVED",
        "thresholds": {
            "WARNING": _threshold_contract("solid_particle_concentration", "solid_particle", "mol/m3", "domain-specific"),
            "HIGH": _threshold_contract("solid_particle_concentration", "solid_particle", "mol/m3", "domain-specific"),
            "CRITICAL": _threshold_contract("solid_particle_concentration", "solid_particle", "mol/m3", "domain-specific")
        }
    },
    "electrolyte_concentration": {
        "monitored_variable": "electrolyte_concentration",
        "status": "UNRESOLVED",
        "thresholds": {
            "WARNING": _threshold_contract("electrolyte_concentration", "electrolyte", "mol/m3", "domain-specific"),
            "HIGH": _threshold_contract("electrolyte_concentration", "electrolyte", "mol/m3", "domain-specific"),
            "CRITICAL": _threshold_contract("electrolyte_concentration", "electrolyte", "mol/m3", "domain-specific")
        }
    },
    "degradation_soh": {
        "monitored_variable": "capacity_retention_state",
        "status": "UNRESOLVED",
        "thresholds": {
            "WARNING": _threshold_contract("capacity_retention_state", "cell", "fraction", "lower"),
            "HIGH": _threshold_contract("capacity_retention_state", "cell", "fraction", "lower"),
            "CRITICAL": _threshold_contract("capacity_retention_state", "cell", "fraction", "lower")
        }
    }
}
assert RISK_SPEC["operational_status"] == "UNASSESSED_UNTIL_VALIDATED"
assert set(RISK_SPEC["domains"]) == {
    "thermal", "mechanical_radial", "mechanical_tangential",
    "solid_concentration", "electrolyte_concentration", "degradation_soh"
}
print("Cell 19: PASS — six-domain risk contract defined; operational thresholds remain UNRESOLVED.")


Cell 19: PASS — six-domain risk contract defined; operational thresholds remain UNRESOLVED.


In [18]:
# Cell 20: Proposed Decision Action Rules & Early-Warning Metrics
ACTION_RULES = {
    "status": "PROPOSED_NOT_OPERATIONAL",
    "activation_condition": "Only applicable after risk thresholds, precedence, and boundary definitions are validated.",
    "rules": {
        "NORMAL": {"action": "CONTINUE_NOMINAL_OPERATION", "status": "PROPOSED"},
        "WARNING": {"action": "EVALUATE_COOLING_AND_CHARGE_DERATING", "status": "PROPOSED"},
        "HIGH": {"action": "EXECUTE_PROACTIVE_CHARGE_DERATE", "status": "PROPOSED"},
        "CRITICAL": {"action": "TERMINATE_CHARGE", "status": "PROPOSED"}
    }
}

EARLY_WARNING_SPEC = {
    "comparison_case": "S5",
    "conventional_alarm_definition": "Alarm time is the earliest time a validated conventional measurement threshold is crossed.",
    "predictive_alarm_definition": "Alarm time is the earliest forecast time at which a validated physical boundary is predicted to be crossed under the declared event rule.",
    "lead_time_definition": "conventional_alarm_time - predictive_alarm_time",
    "event_matching": "A predicted event matches a ground-truth event when both refer to the same domain, cell, and threshold crossing within a declared time tolerance.",
    "no_event_handling": "If no ground-truth event occurs, alarm classification follows an explicitly declared no-event rule; lead time is UNDEFINED when no breach occurs.",
    "metrics": [
        {"name": "warning_lead_time_seconds", "formula": "t_conventional_alarm - t_predictive_alarm"},
        {"name": "false_alarm_rate", "formula": "false_alarms / applicable_negative_evaluations"},
        {"name": "missed_event_rate", "formula": "missed_events / applicable_positive_events"},
        {"name": "threshold_time_error_seconds", "formula": "abs(t_predicted_crossing - t_actual_crossing)"},
        {"name": "physics_residual", "formula": "declared_physics_residual_metric"},
        {"name": "inference_latency_ms", "formula": "elapsed_milliseconds_per_forecast_inference"}
    ],
    "alarm_precedence": "Earliest valid violation within the declared domain/cell/threshold precedence hierarchy."
}
print("Cell 20: PASS — action rules are explicitly proposed/non-operational and early-warning semantics are defined.")


Cell 20: PASS — action rules are explicitly proposed/non-operational and early-warning semantics are defined.


In [19]:
# Cell 21: Formal Hypotheses & Proposed Acceptance Criteria
SCIENTIFIC_OBJECTIVES = [
    {
        "scenario_id": "S1",
        "status": "HYPOTHESIS",
        "scientific_hypothesis": "The V1 surrogate should reproduce held-out nominal trajectories within predefined prediction-error tolerances.",
        "proposed_acceptance_criteria": "Proposed target: solid-particle concentration relative L2 error <= 2% on the declared held-out evaluation set; other outputs require separately declared tolerances.",
        "validation_phase": "Notebook 05/06",
        "evidence_required": "Held-out PyBaMM ground truth and documented evaluation procedure."
    },
    {
        "scenario_id": "S2",
        "status": "HYPOTHESIS",
        "scientific_hypothesis": "Changing charging severity should measurably alter relevant thermal/electrochemical/mechanical trajectories.",
        "proposed_acceptance_criteria": "Effect size and direction must be estimated from matched PyBaMM cases; no monotonicity claim is assumed in Notebook 03.",
        "validation_phase": "Notebook 04/05",
        "evidence_required": "Matched scenarios with controlled C-rate changes."
    },
    {
        "scenario_id": "S3",
        "status": "HYPOTHESIS",
        "scientific_hypothesis": "Changing ambient temperature should measurably alter thermal trajectories under otherwise matched conditions.",
        "proposed_acceptance_criteria": "Observed change must be quantified against a matched healthy baseline; no linearity claim is assumed.",
        "validation_phase": "Notebook 04/05",
        "evidence_required": "Matched ambient-temperature simulations."
    },
    {
        "scenario_id": "S4",
        "status": "HYPOTHESIS",
        "scientific_hypothesis": "Localized Cell-37 degradation should produce measurable divergence from matched healthy cells before or around a declared pack-level event.",
        "proposed_acceptance_criteria": "Test whether Cell 37 divergence exceeds a statistically defined healthy-reference envelope; any 3-sigma rule is proposed, not established.",
        "validation_phase": "Notebook 07",
        "evidence_required": "Repeated healthy/degraded cases and an explicit reference-distribution estimation method."
    },
    {
        "scenario_id": "S5",
        "status": "HYPOTHESIS",
        "scientific_hypothesis": "Combined degradation, ambient stress, and high charging can produce a measurable change in predicted safety-envelope trajectories.",
        "proposed_acceptance_criteria": "Threshold-crossing time error target <= 10% is proposed and must be tested against numerical ground truth.",
        "validation_phase": "Notebook 07",
        "evidence_required": "Validated boundaries and matched simulation ground truth."
    },
    {
        "scenario_id": "S6",
        "status": "HYPOTHESIS",
        "scientific_hypothesis": "A counterfactual reduction in charging rate may restore or improve validated safety margins.",
        "proposed_acceptance_criteria": "Select the highest tested candidate satisfying all validated constraints; otherwise return NO_FEASIBLE_TESTED_CANDIDATE.",
        "validation_phase": "Notebook 09",
        "evidence_required": "Full-horizon candidate simulations and validated constraints."
    }
]
scientific_obj_df = pd.DataFrame(SCIENTIFIC_OBJECTIVES)
print("Cell 21: PASS — hypotheses/acceptance criteria are explicitly proposed and evidence-dependent.")
display(scientific_obj_df)


Cell 21: PASS — hypotheses/acceptance criteria are explicitly proposed and evidence-dependent.


,scenario_id,status,scientific_hypothesis,proposed_acceptance_criteria,validation_phase,evidence_required
0,S1,HYPOTHESIS,The V1 surrogate should reproduce held-out nom...,Proposed target: solid-particle concentration ...,Notebook 05/06,Held-out PyBaMM ground truth and documented ev...
1,S2,HYPOTHESIS,Changing charging severity should measurably a...,Effect size and direction must be estimated fr...,Notebook 04/05,Matched scenarios with controlled C-rate changes.
2,S3,HYPOTHESIS,Changing ambient temperature should measurably...,Observed change must be quantified against a m...,Notebook 04/05,Matched ambient-temperature simulations.
3,S4,HYPOTHESIS,Localized Cell-37 degradation should produce m...,Test whether Cell 37 divergence exceeds a stat...,Notebook 07,Repeated healthy/degraded cases and an explici...
4,S5,HYPOTHESIS,"Combined degradation, ambient stress, and high...",Threshold-crossing time error target <= 10% is...,Notebook 07,Validated boundaries and matched simulation gr...
5,S6,HYPOTHESIS,A counterfactual reduction in charging rate ma...,Select the highest tested candidate satisfying...,Notebook 09,Full-horizon candidate simulations and validat...


In [20]:
# Cell 22: Provenance Register
PROVENANCE_REGISTER = [
    {
        "item": "V0 dataset and metadata",
        "status": "VERIFIED_FROM_ARTIFACTS",
        "evidence": "Exact dataset schema, row count, condition count, split counts, metadata linkage, and SHA-256 hashes verified in Cells 6–7."
    },
    {
        "item": "Virtual pack size = 100 cells",
        "status": "DESIGN_REQUIREMENT",
        "evidence": "Notebook 03 architecture requirement; not evidence that a physical 100-cell pack has been simulated."
    },
    {
        "item": "Target Cell 37",
        "status": "DESIGN_REQUIREMENT",
        "evidence": "Notebook 03 experiment/scenario requirement."
    },
    {
        "item": "Cell 37 resistance multiplier = 1.8",
        "status": "PROVISIONAL",
        "evidence": "Design placeholder; no authoritative physical source asserted in Notebook 03."
    },
    {
        "item": "Cell 37 capacity retention = 0.82",
        "status": "PROVISIONAL",
        "evidence": "Design placeholder; no authoritative physical source asserted in Notebook 03."
    },
    {
        "item": "Cell 37 thermal conductivity factor = 0.85",
        "status": "PROVISIONAL",
        "evidence": "Design placeholder; no authoritative physical source asserted in Notebook 03."
    },
    {
        "item": "Pack nominal voltage",
        "status": "UNRESOLVED",
        "evidence": "Must be determined from selected chemistry/topology in Notebook 04."
    },
    {
        "item": "Pack nominal capacity",
        "status": "UNRESOLVED",
        "evidence": "Must be determined from selected chemistry/topology in Notebook 04."
    },
    {
        "item": "Operational risk thresholds",
        "status": "UNRESOLVED",
        "evidence": "Draft traceability values are inactive; validated values must be sourced and tested downstream."
    },
    {
        "item": "Trained V1 PINN weights",
        "status": "NOT_YET_AVAILABLE",
        "evidence": "Training is outside Notebook 03."
    }
]
provenance_df = pd.DataFrame(PROVENANCE_REGISTER)
print("Cell 22: PASS — provenance statuses distinguish verified artifacts, design requirements, provisional values, and unresolved physics.")
display(provenance_df)


Cell 22: PASS — provenance statuses distinguish verified artifacts, design requirements, provisional values, and unresolved physics.


,item,status,evidence
0,V0 dataset and metadata,VERIFIED_FROM_ARTIFACTS,"Exact dataset schema, row count, condition cou..."
1,Virtual pack size = 100 cells,DESIGN_REQUIREMENT,Notebook 03 architecture requirement; not evid...
2,Target Cell 37,DESIGN_REQUIREMENT,Notebook 03 experiment/scenario requirement.
3,Cell 37 resistance multiplier = 1.8,PROVISIONAL,Design placeholder; no authoritative physical ...
4,Cell 37 capacity retention = 0.82,PROVISIONAL,Design placeholder; no authoritative physical ...
5,Cell 37 thermal conductivity factor = 0.85,PROVISIONAL,Design placeholder; no authoritative physical ...
6,Pack nominal voltage,UNRESOLVED,Must be determined from selected chemistry/top...
7,Pack nominal capacity,UNRESOLVED,Must be determined from selected chemistry/top...
8,Operational risk thresholds,UNRESOLVED,Draft traceability values are inactive; valida...
9,Trained V1 PINN weights,NOT_YET_AVAILABLE,Training is outside Notebook 03.


In [21]:
# Cell 23: Assemble Master V1 Specification

V1_SPECIFICATION = {
    "version": "V1",
    "project": "Physics-Informed Predictive Battery Digital Twin",
    "timestamp": datetime.datetime.utcnow().isoformat() + "Z",
    "v0_lineage": V0_LINEAGE,
    "pack": PACK_CONFIG,
    "input_specification": INPUT_SPEC,
    "output_specification": OUTPUT_SPEC,
    "degradation_contract": DEGRADATION_CONTRACT,
    "cell37_degradation_contract": CELL37_DEGRADATION_CONTRACT,
    "scenarios": SCENARIOS,
    "s6_intervention_sweep": S6_INTERVENTION_SWEEP,
    "scenario_matrix": matrix_rows,
    "cell_localization_experiment": CELL_LOCALIZATION_EXPERIMENT,
    "cell37_experiment": CELL37_EXPERIMENT,
    "baselines": BASELINE_SPEC,
    "risk_spec": RISK_SPEC,
    "action_rules": ACTION_RULES,
    "early_warning_spec": EARLY_WARNING_SPEC,
    "scientific_objectives": SCIENTIFIC_OBJECTIVES,
    "provenance_register": PROVENANCE_REGISTER,
    "boundary_declarations": {
        "no_training_in_notebook_03": True,
        "no_frontend_or_3d_in_notebook_03": True,
        "no_fabricated_predictions": True,
        "operational_risk_status": "UNASSESSED_UNTIL_VALIDATED",
        "v0_is_single_cell_continuum": True,
        "v1_has_100_cell_dynamic_localization_contract": True,
        "cell37_is_benchmark_not_permanent_fault": True,
        "fault_location_must_be_inferred_from_cell_level_evidence": True,
        "random_fault_assignment_for_final_results_forbidden": True
    }
}

print("Cell 23: PASS — master specification assembled with whole-pack dynamic localization contract.")


Cell 23: PASS — master specification assembled with whole-pack dynamic localization contract.


/tmp/ipykernel_1527/2417440786.py:6: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "timestamp": datetime.datetime.utcnow().isoformat() + "Z",


In [42]:
# Cell 24: SINGLE AUTHORITATIVE V1 SPECIFICATION VALIDATOR

def validate_specification(spec):

    errors = []
    warnings = []

    def err(message):
        errors.append(message)

    # =========================================================
    # 1. VERSION + V0 LINEAGE
    # =========================================================

    if spec.get("version") != "V1":
        err("version must be V1")

    lineage = spec.get("v0_lineage", {})

    if lineage.get("status") != "VERIFIED":
        err("V0 lineage must be VERIFIED")

    if lineage.get("mock_pinn") is True:
        err("mock_pinn=True is forbidden")

    if lineage.get("shape") != [43200, 16]:
        err("V0 shape must be [43200, 16]")

    if lineage.get("columns") != EXPECTED_V0_COLUMNS:
        err("V0 exact schema mismatch")

    if lineage.get("condition_count") != 20:
        err("V0 must contain exactly 20 conditions")

    if lineage.get("split_condition_counts") != {
        "train": 12,
        "val": 6,
        "test": 2
    }:
        err("V0 condition split must be 12/6/2")

    # =========================================================
    # 2. 100-CELL PACK
    # =========================================================

    pack = spec.get("pack", {})

    if pack.get("series_cells") != 100:
        err("pack must contain exactly 100 cells")

    if pack.get("monitored_cells") != list(range(1, 101)):
        err("monitored_cells must be exactly 1..100")

    if pack.get("reference_experiment_cell_id") != 37:
        err("Cell 37 must remain the benchmark experiment cell")

    localization = pack.get("fault_localization", {})

    if localization.get("mode") != "DYNAMIC_WHOLE_PACK":
        err("fault localization mode must be DYNAMIC_WHOLE_PACK")

    if localization.get("candidate_cells") != list(range(1, 101)):
        err("fault localization candidates must be all 100 cells")

    if localization.get("random_fault_assignment") is not False:
        err("random fault assignment must be forbidden")

    for key in [
        "parallel_strings",
        "nominal_voltage_v",
        "nominal_capacity_ah"
    ]:
        obj = pack.get(key, {})

        if not isinstance(obj, dict):
            err(f"{key} must be represented as an object")
            continue

        if obj.get("value") is not None:
            err(
                f"{key} must remain value=None until physically sourced"
            )

        if obj.get("status") != "UNRESOLVED":
            err(
                f"{key} must remain UNRESOLVED"
            )

    # =========================================================
    # 3. INPUT SPECIFICATION
    # =========================================================

    allowed_classifications = {
        "measured",
        "controlled",
        "estimated",
        "latent",
        "training_label"
    }

    required_input_fields = {
        "cell_id",
        "time_s",
        "operating_state",
        "pack_voltage_v",
        "pack_current_a",
        "charging_state",
        "charging_rate_c",
        "ambient_temp_c",
        "surface_temperature_c",
        "current_soc",
        "initial_soc",
        "cycle_number",
        "operating_history",
        "history_window_steps",
        "capacity_state",
        "resistance_state",
        "degradation_state",
        "cooling_flow_state",
        "spatial_r_norm"
    }

    input_rows = spec.get(
        "input_specification",
        []
    )

    actual_input_fields = {
        row.get("field")
        for row in input_rows
        if isinstance(row, dict)
    }

    missing_inputs = (
        required_input_fields - actual_input_fields
    )

    if missing_inputs:
        err(
            f"missing INPUT_SPEC fields: "
            f"{sorted(missing_inputs)}"
        )

    for row in input_rows:

        if not isinstance(row, dict):
            err(
                "every INPUT_SPEC entry must be a dictionary"
            )
            continue

        for key in [
            "field",
            "type",
            "unit",
            "physical_meaning",
            "source",
            "scope",
            "valid_range",
            "availability",
            "classification"
        ]:
            if key not in row:
                err(
                    f"input field {row.get('field')} "
                    f"missing {key}"
                )

        if row.get("valid_range") is None:
            err(
                f"input field {row.get('field')} "
                f"has no valid_range"
            )

        if row.get("classification") not in allowed_classifications:
            err(
                f"input field {row.get('field')} "
                f"has invalid classification"
            )

    # =========================================================
    # 4. OUTPUT SPECIFICATION
    # =========================================================

    required_output_fields = {
        "solid_particle_concentration",
        "electrolyte_concentration",
        "cell_temperature",
        "radial_stress",
        "tangential_stress",
        "capacity_retention_state",
        "resistance_multiplier_state",
        "soh_definition",
        "future_solid_particle_concentration_trajectory",
        "future_electrolyte_concentration_trajectory",
        "future_temperature_trajectory",
        "future_radial_stress_trajectory",
        "future_tangential_stress_trajectory",
        "future_soh_trajectory",
        "per_cell_health_state",
        "per_cell_health_score",
        "per_cell_anomaly_score",
        "defective_cells",
        "affected_cells",
        "potentially_affected_cells",
        "healthy_cells",
        "cell_localization_confidence",
        "localization_evidence",
        "localization_trigger",
        "safety_envelope_margin",
        "time_to_threshold_s",
        "rul_projection_cycles",
        "rul_eol_criterion",
        "rul_future_use_assumption",
        "risk_level",
        "recommended_action"
    }

    output_rows = spec.get(
        "output_specification",
        []
    )

    actual_output_fields = {
        row.get("field")
        for row in output_rows
        if isinstance(row, dict)
    }

    missing_outputs = (
        required_output_fields - actual_output_fields
    )

    if missing_outputs:
        err(
            f"missing OUTPUT_SPEC fields: "
            f"{sorted(missing_outputs)}"
        )

    for row in output_rows:

        if not isinstance(row, dict):
            err(
                "every OUTPUT_SPEC entry must be a dictionary"
            )
            continue

        field = row.get("field")

        if (
            field == "solid_particle_concentration"
            and row.get("domain") != "solid_particle"
        ):
            err(
                "solid concentration domain must be solid_particle"
            )

        if (
            field == "electrolyte_concentration"
            and row.get("domain") != "electrolyte"
        ):
            err(
                "electrolyte concentration domain must be electrolyte"
            )

        if (
            field == "radial_stress"
            and row.get("coordinate") != "radial"
        ):
            err(
                "radial stress coordinate must be radial"
            )

        if (
            field == "tangential_stress"
            and row.get("coordinate") != "tangential/hoop"
        ):
            err(
                "tangential stress coordinate must be tangential/hoop"
            )

    # =========================================================
    # 5. CELL-AGNOSTIC DEGRADATION CONTRACT
    # =========================================================

    degradation = spec.get(
        "degradation_contract",
        {}
    )

    if (
        degradation.get("candidate_cells")
        != list(range(1, 101))
    ):
        err(
            "degradation candidate cells must be all 100 cells"
        )

    if (
        degradation.get("benchmark", {}).get("cell_id")
        != 37
    ):
        err(
            "degradation benchmark must be Cell 37"
        )

    if (
        degradation.get("random_fault_assignment")
        is not False
    ):
        err(
            "random fault assignment is forbidden"
        )

    # =========================================================
    # 6. SCENARIOS
    # =========================================================

    scenarios = spec.get(
        "scenarios",
        []
    )

    expected_ids = [
        "S1",
        "S2",
        "S3",
        "S4",
        "S5",
        "S6"
    ]

    ids = [
        s.get("scenario_id")
        for s in scenarios
        if isinstance(s, dict)
    ]

    if ids != expected_ids:
        err(
            "scenario IDs must be exactly S1-S6"
        )

    smap = {
        s.get("scenario_id"): s
        for s in scenarios
        if isinstance(s, dict)
    }

    s1 = smap.get("S1", {})
    s2 = smap.get("S2", {})
    s3 = smap.get("S3", {})
    s4 = smap.get("S4", {})
    s5 = smap.get("S5", {})
    s6 = smap.get("S6", {})

    # ---------------------------------------------------------
    # S1
    # ---------------------------------------------------------

    if s1.get(
        "affected_cells_mode"
    ) != "NONE_EXPECTED":
        err(
            "S1 must use NONE_EXPECTED affected-cell mode"
        )

    # ---------------------------------------------------------
    # S2
    # ---------------------------------------------------------

    if s2.get(
        "affected_cells_mode"
    ) != "AUTO_DETECT":
        err(
            "S2 must scan all cells using AUTO_DETECT"
        )

    if (
        s2.get("inputs", {}).get("ambient_temp_c")
        != s1.get("inputs", {}).get("ambient_temp_c")
    ):
        err(
            "S2 must preserve S1 ambient temperature"
        )

    if (
        s2.get("inputs", {}).get("initial_soc")
        != s1.get("inputs", {}).get("initial_soc")
    ):
        err(
            "S2 must preserve S1 initial SOC"
        )

    if (
        s2.get("inputs", {}).get("target_charge_rate_c")
        == s1.get("inputs", {}).get("target_charge_rate_c")
    ):
        err(
            "S2 must change charging rate relative to S1"
        )

    # ---------------------------------------------------------
    # S3
    # ---------------------------------------------------------

    if s3.get(
        "affected_cells_mode"
    ) != "AUTO_DETECT":
        err(
            "S3 must scan all cells using AUTO_DETECT"
        )

    if (
        s3.get("inputs", {}).get("target_charge_rate_c")
        != s1.get("inputs", {}).get("target_charge_rate_c")
    ):
        err(
            "S3 must preserve S1 charge rate"
        )

    if (
        s3.get("inputs", {}).get("initial_soc")
        != s1.get("inputs", {}).get("initial_soc")
    ):
        err(
            "S3 must preserve S1 initial SOC"
        )

    if (
        s3.get("inputs", {}).get("ambient_temp_c")
        == s1.get("inputs", {}).get("ambient_temp_c")
    ):
        err(
            "S3 must change ambient temperature relative to S1"
        )

    # ---------------------------------------------------------
    # S4
    # ---------------------------------------------------------

    if s4.get("target_cell") is not None:
        err(
            "S4 must not permanently hard-code one target cell"
        )

    if s4.get(
        "affected_cells_mode"
    ) not in {
        "DECLARED_PER_RUN",
        "AUTO_DETECT"
    }:
        err(
            "S4 must use a dynamic/non-hard-coded cell-selection mode"
        )

    if s4.get(
        "benchmark_experiment_cell"
    ) != 37:
        err(
            "S4 must retain Cell 37 only as the benchmark"
        )

    # ---------------------------------------------------------
    # S5
    # ---------------------------------------------------------

    if s5.get("target_cell") is not None:
        err(
            "S5 must not permanently hard-code one target cell"
        )

    if s5.get(
        "affected_cells_mode"
    ) not in {
        "INHERIT_OR_DECLARE_PER_RUN",
        "DECLARED_PER_RUN",
        "AUTO_DETECT"
    }:
        err(
            "S5 must use a dynamic/non-hard-coded cell-selection mode"
        )

    if s5.get(
        "benchmark_experiment_cell"
    ) != 37:
        err(
            "S5 must retain Cell 37 only as the benchmark"
        )

    if (
        s5.get("changes")
        != s4.get("changes")
    ):
        err(
            "S5 must inherit S4 degradation configuration"
        )

    if (
        s5.get("inputs", {}).get("ambient_temp_c")
        == s4.get("inputs", {}).get("ambient_temp_c")
    ):
        err(
            "S5 must change ambient temperature relative to S4"
        )

    if (
        s5.get("inputs", {}).get("target_charge_rate_c")
        == s4.get("inputs", {}).get("target_charge_rate_c")
    ):
        err(
            "S5 must change charging rate relative to S4"
        )

    # ---------------------------------------------------------
    # S6
    # ---------------------------------------------------------

    if s6.get("target_cell") is not None:
        err(
            "S6 must not hard-code one target cell"
        )

    if s6.get(
        "affected_cells_mode"
    ) != "INHERIT_FROM_S5":
        err(
            "S6 must inherit affected-cell state from S5"
        )

    if (
        s6.get("changes")
        != s5.get("changes")
    ):
        err(
            "S6 must preserve complete S5 degradation state"
        )

    if (
        s6.get("inputs", {}).get("ambient_temp_c")
        != s5.get("inputs", {}).get("ambient_temp_c")
    ):
        err(
            "S6 must preserve S5 ambient temperature"
        )

    if (
        s6.get("inputs", {}).get("initial_soc")
        != s5.get("inputs", {}).get("initial_soc")
    ):
        err(
            "S6 must preserve S5 initial SOC"
        )

    if (
        s6.get("inputs", {}).get("target_charge_rate_c")
        == s5.get("inputs", {}).get("target_charge_rate_c")
    ):
        err(
            "S6 must change the intervention C-rate"
        )

    # =========================================================
    # 7. S6 SWEEP
    # =========================================================

    sweep = spec.get(
        "s6_intervention_sweep",
        {}
    )

    candidates = sweep.get(
        "tested_candidate_c_rates",
        []
    )

    expected_candidates = [
        0.4,
        0.8,
        1.0,
        1.5,
        2.0
    ]

    if sorted(candidates) != expected_candidates:
        err(
            "S6 candidates must be exactly "
            "0.4, 0.8, 1.0, 1.5, 2.0 C"
        )

    if (
        sweep.get("winner_preselected")
        is not None
    ):
        err(
            "S6 winner must not be preselected"
        )

    if (
        sweep.get("candidate_cells_scanned_each_run")
        != list(range(1, 101))
    ):
        err(
            "S6 must scan all 100 cells for every candidate"
        )

    if sweep.get(
        "base_scenario"
    ) != "S5":
        err(
            "S6 base scenario must be S5"
        )

    # =========================================================
    # 8. SCENARIO MATRIX
    # =========================================================

    matrix = spec.get(
        "scenario_matrix",
        []
    )

    if not isinstance(matrix, list):
        err(
            "scenario_matrix must be a list"
        )

    elif len(matrix) != 10:
        err(
            "scenario matrix must contain exactly 10 rows"
        )

    else:

        matrix_cell_counts = [
            row.get("cells_scanned")
            for row in matrix
            if isinstance(row, dict)
        ]

        if not all(
            value == 100
            for value in matrix_cell_counts
        ):
            err(
                "every scenario matrix row must scan all 100 cells"
            )

        matrix_s6_rates = sorted(
            row.get("charge_rate_c")
            for row in matrix
            if isinstance(row, dict)
            and row.get("scenario_id") == "S6"
        )

        if matrix_s6_rates != expected_candidates:
            err(
                "scenario matrix S6 candidate rates mismatch"
            )

    # =========================================================
    # 9. WHOLE-PACK LOCALIZATION EXPERIMENT
    # =========================================================

    localization_experiment = spec.get(
        "cell_localization_experiment",
        {}
    )

    if (
        localization_experiment.get("candidate_cells")
        != list(range(1, 101))
    ):
        err(
            "localization experiment must cover all 100 cells"
        )

    if (
        localization_experiment.get("benchmark_cell")
        != 37
    ):
        err(
            "localization benchmark must be Cell 37"
        )

    if (
        localization_experiment.get(
            "no_hard_coded_fault_location"
        ) is not True
    ):
        err(
            "hard-coded fault localization is forbidden"
        )

    if (
        localization_experiment.get(
            "no_random_fault_location"
        ) is not True
    ):
        err(
            "random fault localization is forbidden"
        )

    # =========================================================
    # 10. CELL 37 CONTROLLED BENCHMARK
    # =========================================================

    cell37_experiment = spec.get(
        "cell37_experiment",
        {}
    )

    if (
        cell37_experiment.get("target_cell")
        != 37
    ):
        err(
            "Cell 37 benchmark experiment is missing"
        )

    # =========================================================
    # 11. B0-B3
    # =========================================================

    baselines = {
        b.get("model_id"): b.get("name")
        for b in spec.get("baselines", [])
        if isinstance(b, dict)
    }

    expected_baselines = {
        "B0": "Conventional Threshold Monitoring",
        "B1": "Data-Only Neural Network",
        "B2": "Existing V0 PINN",
        "B3": "Proposed V1 Predictive PINN Digital Twin"
    }

    for model_id, expected_name in expected_baselines.items():

        if baselines.get(model_id) != expected_name:
            err(
                f"{model_id} baseline mapping is incorrect"
            )

    # =========================================================
    # 12. RISK ENGINE
    # =========================================================

    risk = spec.get(
        "risk_spec",
        {}
    )

    if (
        risk.get("operational_status")
        != "UNASSESSED_UNTIL_VALIDATED"
    ):
        err(
            "risk must remain UNASSESSED_UNTIL_VALIDATED"
        )

    required_risk_domains = {
        "thermal",
        "mechanical_radial",
        "mechanical_tangential",
        "solid_concentration",
        "electrolyte_concentration",
        "degradation_soh"
    }

    if (
        set(risk.get("domains", {}))
        != required_risk_domains
    ):
        err(
            "risk domains must contain all six required domains"
        )

    required_threshold_keys = {
        "quantity",
        "domain",
        "value",
        "unit",
        "source",
        "chemistry_applicability",
        "model_applicability",
        "rationale",
        "validation_status",
        "boundary_rule",
        "precedence"
    }

    for domain_name, domain in risk.get(
        "domains",
        {}
    ).items():

        if domain.get(
            "status"
        ) != "UNRESOLVED":
            err(
                f"risk domain {domain_name} "
                f"must remain UNRESOLVED"
            )

        for level in [
            "WARNING",
            "HIGH",
            "CRITICAL"
        ]:

            threshold = domain.get(
                "thresholds",
                {}
            ).get(
                level,
                {}
            )

            missing_keys = (
                required_threshold_keys
                - set(threshold)
            )

            if missing_keys:
                err(
                    f"risk threshold "
                    f"{domain_name}/{level} "
                    f"missing {sorted(missing_keys)}"
                )

            status = threshold.get(
                "validation_status"
            )

            if status == "VALIDATED":

                if not threshold.get("source"):
                    err(
                        f"validated threshold "
                        f"{domain_name}/{level} "
                        f"requires a source"
                    )

                if threshold.get("value") is None:
                    err(
                        f"validated threshold "
                        f"{domain_name}/{level} "
                        f"requires a value"
                    )

            elif status == "UNRESOLVED":

                if threshold.get(
                    "value"
                ) is not None:
                    err(
                        f"unresolved threshold "
                        f"{domain_name}/{level} "
                        f"must use value=None"
                    )

    # =========================================================
    # 13. EARLY WARNING
    # =========================================================

    early_warning = spec.get(
        "early_warning_spec",
        {}
    )

    required_ew_fields = [
        "conventional_alarm_definition",
        "predictive_alarm_definition",
        "lead_time_definition",
        "event_matching",
        "no_event_handling",
        "alarm_precedence",
        "metrics"
    ]

    for field in required_ew_fields:

        if not early_warning.get(field):
            err(
                f"early-warning field missing: {field}"
            )

    # =========================================================
    # 14. BOUNDARY DECLARATIONS
    # =========================================================

    boundaries = spec.get(
        "boundary_declarations",
        {}
    )

    if (
        boundaries.get(
            "cell37_is_benchmark_not_permanent_fault"
        ) is not True
    ):
        err(
            "Cell 37 must explicitly be benchmark-only"
        )

    if (
        boundaries.get(
            "fault_location_must_be_inferred_from_cell_level_evidence"
        ) is not True
    ):
        err(
            "fault location must be inferred from cell-level evidence"
        )

    if (
        boundaries.get(
            "random_fault_assignment_for_final_results_forbidden"
        ) is not True
    ):
        err(
            "random fault assignment must be forbidden"
        )

    if (
        boundaries.get(
            "no_fabricated_predictions"
        ) is not True
    ):
        err(
            "no_fabricated_predictions must be True"
        )

    # =========================================================
    # 15. FABRICATION GUARD
    # =========================================================

    forbidden_result_keys = {
        "results",
        "predictions",
        "evaluation_results",
        "trained_model_results",
        "fabricated_results"
    }

    for key in forbidden_result_keys:

        if key in spec:
            err(
                f"forbidden result container present: {key}"
            )

    # =========================================================
    # FINAL VALIDATION RESULT
    # =========================================================

    return {
        "valid": len(errors) == 0,
        "errors": errors,
        "warnings": warnings
    }


print(
    "Cell 24: PASS — "
    "single authoritative whole-pack localization validator registered."
)

Cell 24: PASS — single authoritative whole-pack localization validator registered.


In [43]:
# Cell 24B: Duplicate Validator Guard
# This cell intentionally contains no second validator implementation.
assert callable(validate_specification), "Authoritative validator missing."
print("Cell 24B: PASS — exactly one validator implementation is authoritative.")


Cell 24B: PASS — exactly one validator implementation is authoritative.


In [44]:
# Cell 25: Negative-Test Fixture Generator for Whole-Pack Localization

def make_mutated_fixture(tag):
    mutated = copy.deepcopy(V1_SPECIFICATION)

    # ------------------------------------------------------------
    # Pack / localization integrity
    # ------------------------------------------------------------
    if tag == "96_cells":
        mutated["pack"]["monitored_cells"] = list(range(1, 97))

    elif tag == "duplicate_cell_id":
        mutated["pack"]["monitored_cells"] = list(range(1, 100)) + [99]

    elif tag == "missing_cell_id":
        mutated["pack"]["monitored_cells"] = [
            i for i in range(1, 101) if i != 50
        ]

    elif tag == "malformed_cell_id":
        cells = list(range(1, 101))
        cells[10] = "CELL_X"
        mutated["pack"]["monitored_cells"] = cells

    elif tag == "missing_cell37_benchmark":
        mutated["pack"]["reference_experiment_cell_id"] = 38

    elif tag == "non_dynamic_localization":
        mutated["pack"]["fault_localization"]["mode"] = "CELL37_ONLY"

    elif tag == "random_fault_assignment":
        mutated["pack"]["fault_localization"]["random_fault_assignment"] = True

    # ------------------------------------------------------------
    # Output specification integrity
    # ------------------------------------------------------------
    elif tag == "missing_solid_concentration":
        mutated["output_specification"] = [
            r
            for r in mutated["output_specification"]
            if r.get("field") != "solid_particle_concentration"
        ]

    elif tag == "missing_electrolyte_concentration":
        mutated["output_specification"] = [
            r
            for r in mutated["output_specification"]
            if r.get("field") != "electrolyte_concentration"
        ]

    elif tag == "missing_radial_stress":
        mutated["output_specification"] = [
            r
            for r in mutated["output_specification"]
            if r.get("field") != "radial_stress"
        ]

    elif tag == "missing_tangential_stress":
        mutated["output_specification"] = [
            r
            for r in mutated["output_specification"]
            if r.get("field") != "tangential_stress"
        ]

    elif tag == "missing_per_cell_health":
        mutated["output_specification"] = [
            r
            for r in mutated["output_specification"]
            if r.get("field") != "per_cell_health_state"
        ]

    elif tag == "missing_defective_cells_output":
        mutated["output_specification"] = [
            r
            for r in mutated["output_specification"]
            if r.get("field") != "defective_cells"
        ]

    elif tag == "missing_affected_cells_output":
        mutated["output_specification"] = [
            r
            for r in mutated["output_specification"]
            if r.get("field") != "affected_cells"
        ]

    elif tag == "missing_potentially_affected_output":
        mutated["output_specification"] = [
            r
            for r in mutated["output_specification"]
            if r.get("field") != "potentially_affected_cells"
        ]

    # ------------------------------------------------------------
    # Cell 37 benchmark
    # ------------------------------------------------------------
    elif tag == "missing_cell37_experiment":
        mutated["cell37_experiment"]["target_cell"] = 38

    # ------------------------------------------------------------
    # Scenario isolation
    # ------------------------------------------------------------
    elif tag == "s2_not_auto_detect":
        mutated["scenarios"][1]["affected_cells_mode"] = "CELL37_ONLY"

    elif tag == "s2_wrong_isolation":
        mutated["scenarios"][1]["inputs"]["ambient_temp_c"] = 30.0

    elif tag == "s3_not_auto_detect":
        mutated["scenarios"][2]["affected_cells_mode"] = "CELL37_ONLY"

    elif tag == "s3_wrong_isolation":
        mutated["scenarios"][2]["inputs"]["target_charge_rate_c"] = 1.5

    # ------------------------------------------------------------
    # Dynamic localization / S4-S6 target rules
    # ------------------------------------------------------------
    elif tag == "s4_hardcoded_target":
        mutated["scenarios"][3]["target_cell"] = 37

    elif tag == "s5_hardcoded_target":
        mutated["scenarios"][4]["target_cell"] = 37

    elif tag == "s6_hardcoded_target":
        mutated["scenarios"][5]["target_cell"] = 37

    elif tag == "s6_changes_affected_cells":
        mutated["scenarios"][5]["affected_cells_mode"] = "DECLARE_NEW_CELL"

    elif tag == "s6_changes_ambient":
        mutated["scenarios"][5]["inputs"]["ambient_temp_c"] = 40.0

    elif tag == "s6_changes_soc":
        mutated["scenarios"][5]["inputs"]["initial_soc"] = 0.5

    elif tag == "s6_changes_degradation":
        mutated["scenarios"][5]["changes"] = {
            "degradation_state": "healthy"
        }

    # ------------------------------------------------------------
    # Scenario ID integrity
    # ------------------------------------------------------------
    elif tag == "duplicate_scenario_id":
        scenarios = copy.deepcopy(mutated["scenarios"])

        # Make S6 have the same ID as S5.
        scenarios[-1]["scenario_id"] = scenarios[-2]["scenario_id"]

        mutated["scenarios"] = scenarios

    elif tag == "missing_scenario_id":
        scenarios = copy.deepcopy(mutated["scenarios"])

        # Remove the scenario_id field from one required scenario.
        scenarios[2].pop("scenario_id", None)

        mutated["scenarios"] = scenarios

    # ------------------------------------------------------------
    # S6 intervention sweep
    # ------------------------------------------------------------
    elif tag == "empty_s6_sweep":
        mutated["s6_intervention_sweep"]["tested_candidate_c_rates"] = []

    elif tag == "identical_s6_candidates":
        mutated["s6_intervention_sweep"][
            "tested_candidate_c_rates"
        ] = [2.0, 2.0, 2.0, 2.0, 2.0]

    elif tag == "no_changed_candidate":
        mutated["s6_intervention_sweep"][
            "tested_candidate_c_rates"
        ] = [2.0, 2.0, 2.0, 2.0, 2.0]

    elif tag == "preselected_s6_winner":
        mutated["s6_intervention_sweep"]["winner_preselected"] = 1.0

    elif tag == "s6_not_all_cells_scanned":
        mutated["s6_intervention_sweep"][
            "candidate_cells_scanned_each_run"
        ] = list(range(1, 51))

    # ------------------------------------------------------------
    # Baselines
    # ------------------------------------------------------------
    elif tag == "wrong_b0_b3":
        mutated["baselines"][0]["name"] = "Data-Only Neural Network"

    # ------------------------------------------------------------
    # Risk specification
    # ------------------------------------------------------------
    elif tag == "risk_validated_without_source":
        mutated["risk_spec"]["domains"]["thermal"]["thresholds"][
            "CRITICAL"
        ]["validation_status"] = "VALIDATED"

    elif tag == "risk_operational":
        mutated["risk_spec"]["operational_status"] = "OPERATIONAL"

    # ------------------------------------------------------------
    # Fabrication / provenance
    # ------------------------------------------------------------
    elif tag == "fabricated_result":
        mutated["predictions"] = {
            "cell_37": "bad"
        }

    elif tag == "mock_v0":
        mutated["v0_lineage"]["mock_pinn"] = True

    # ------------------------------------------------------------
    # Input schema
    # ------------------------------------------------------------
    elif tag == "invalid_input_valid_range":
        mutated["input_specification"][0]["valid_range"] = None

    # ------------------------------------------------------------
    # Required specification sections
    # ------------------------------------------------------------
    elif tag == "missing_early_warning":
        mutated.pop("early_warning_spec", None)

    elif tag == "missing_localization_contract":
        mutated.pop("cell_localization_experiment", None)

    else:
        raise ValueError(
            f"Unknown negative test mutation tag: {tag}"
        )

    return mutated


print(
    "Cell 25: PASS — negative-test fixture generator "
    "registered with explicit scenario-ID integrity cases."
)

Cell 25: PASS — negative-test fixture generator registered.


In [45]:
# Cell 26: Comprehensive Negative Test Suite

negative_cases = [
    # ------------------------------------------------------------
    # Pack / localization
    # ------------------------------------------------------------
    "96_cells",
    "duplicate_cell_id",
    "missing_cell_id",
    "malformed_cell_id",
    "missing_cell37_benchmark",
    "non_dynamic_localization",
    "random_fault_assignment",

    # ------------------------------------------------------------
    # Required outputs
    # ------------------------------------------------------------
    "missing_solid_concentration",
    "missing_electrolyte_concentration",
    "missing_radial_stress",
    "missing_tangential_stress",
    "missing_per_cell_health",
    "missing_defective_cells_output",
    "missing_affected_cells_output",
    "missing_potentially_affected_output",

    # ------------------------------------------------------------
    # Cell 37 benchmark
    # ------------------------------------------------------------
    "missing_cell37_experiment",

    # ------------------------------------------------------------
    # Scenario isolation
    # ------------------------------------------------------------
    "s2_not_auto_detect",
    "s2_wrong_isolation",
    "s3_not_auto_detect",
    "s3_wrong_isolation",

    # ------------------------------------------------------------
    # Dynamic localization target protection
    # ------------------------------------------------------------
    "s4_hardcoded_target",
    "s5_hardcoded_target",
    "s6_hardcoded_target",

    # ------------------------------------------------------------
    # S6 state invariance
    # ------------------------------------------------------------
    "s6_changes_affected_cells",
    "s6_changes_ambient",
    "s6_changes_soc",
    "s6_changes_degradation",

    # ------------------------------------------------------------
    # Scenario ID integrity
    # ------------------------------------------------------------
    "duplicate_scenario_id",
    "missing_scenario_id",

    # ------------------------------------------------------------
    # S6 sweep
    # ------------------------------------------------------------
    "empty_s6_sweep",
    "identical_s6_candidates",
    "no_changed_candidate",
    "preselected_s6_winner",
    "s6_not_all_cells_scanned",

    # ------------------------------------------------------------
    # Baselines
    # ------------------------------------------------------------
    "wrong_b0_b3",

    # ------------------------------------------------------------
    # Risk
    # ------------------------------------------------------------
    "risk_validated_without_source",
    "risk_operational",

    # ------------------------------------------------------------
    # No fabricated / mock results
    # ------------------------------------------------------------
    "fabricated_result",
    "mock_v0",

    # ------------------------------------------------------------
    # Input schema
    # ------------------------------------------------------------
    "invalid_input_valid_range",

    # ------------------------------------------------------------
    # Required sections
    # ------------------------------------------------------------
    "missing_early_warning",
    "missing_localization_contract",
]


negative_rows = []


for tag in negative_cases:

    bad_fixture = make_mutated_fixture(tag)

    try:
        result = validate_specification(
            bad_fixture
        )

        # A valid negative test requires the authoritative
        # validator to reject the corrupted specification.
        rejected = (
            isinstance(result, dict)
            and result.get("valid") is False
        )

        if rejected:
            details = "; ".join(
                result.get("errors", [])[:3]
            )

        else:
            details = (
                "INVALID FIXTURE WAS ACCEPTED BY VALIDATOR"
            )

    except Exception as exc:

        # IMPORTANT:
        # A validator exception is NOT considered success.
        # It means the validator itself needs repair.
        rejected = False

        details = (
            "Validator exception: "
            f"{type(exc).__name__}: {exc}"
        )

    negative_rows.append({
        "test_name": tag,
        "rejected_as_expected": rejected,
        "details": details
    })


negative_df = pd.DataFrame(
    negative_rows
)


failed_tests = negative_df.loc[
    ~negative_df["rejected_as_expected"],
    "test_name"
].tolist()


if failed_tests:

    print(
        "Cell 26: FAIL"
    )

    display(
        negative_df[
            ~negative_df["rejected_as_expected"]
        ]
    )

    raise AssertionError(
        "Negative-test suite failed. "
        "The following invalid fixtures were not "
        "correctly rejected:\n"
        + "\n".join(
            f"- {name}"
            for name in failed_tests
        )
    )


assert len(negative_df) == len(
    negative_cases
), (
    "Negative-test result count does not match "
    "negative-case count."
)


assert negative_df[
    "test_name"
].is_unique, (
    "Negative-test names are not unique."
)


print(
    f"Cell 26: PASS — "
    f"{len(negative_df)}/{len(negative_cases)} "
    f"invalid fixtures rejected by the authoritative validator."
)

display(
    negative_df
)

Cell 26: PASS — 40/40 invalid fixtures rejected.


,test_name,rejected_as_expected,details
0,96_cells,True,monitored_cells must be exactly 1..100
1,duplicate_cell_id,True,monitored_cells must be exactly 1..100
2,missing_cell_id,True,monitored_cells must be exactly 1..100
3,malformed_cell_id,True,monitored_cells must be exactly 1..100
4,missing_cell37_benchmark,True,Cell 37 must remain the benchmark experiment cell
5,non_dynamic_localization,True,fault localization mode must be DYNAMIC_WHOLE_...
6,random_fault_assignment,True,random fault assignment must be forbidden
7,missing_solid_concentration,True,missing OUTPUT_SPEC fields: ['solid_particle_c...
8,missing_electrolyte_concentration,True,missing OUTPUT_SPEC fields: ['electrolyte_conc...
9,missing_radial_stress,True,missing OUTPUT_SPEC fields: ['radial_stress']


In [46]:
# Cell 27: Positive Specification Validation

positive_validation = validate_specification(
    V1_SPECIFICATION
)

if not positive_validation["valid"]:

    print("Cell 27: FAIL — Positive validation failed.")

    for error in positive_validation["errors"]:
        print("  -", error)

    raise AssertionError(
        "Positive validation failed:\n"
        + "\n".join(
            positive_validation["errors"]
        )
    )

print(
    "Cell 27: PASS — Positive specification validation passed."
)

print(
    "  Validator errors:",
    len(positive_validation["errors"])
)

print(
    "  Validator warnings:",
    len(positive_validation["warnings"])
)

Cell 27: PASS — Positive specification validation passed.
  Validator errors: 0
  Validator warnings: 0


In [47]:
# Cell 28: Core Artifact Export

SCENARIO_DIR.mkdir(
    parents=True,
    exist_ok=True
)

master_json_path = (
    SCENARIO_DIR / "v1_scenario_specification.json"
)

matrix_csv_path = (
    SCENARIO_DIR / "v1_scenario_matrix.csv"
)

input_csv_path = (
    SCENARIO_DIR / "v1_input_specification.csv"
)

output_csv_path = (
    SCENARIO_DIR / "v1_output_specification.csv"
)

validation_report_path = (
    SCENARIO_DIR / "validation_report.json"
)

execution_log_path = (
    SCENARIO_DIR / "execution_log.md"
)

next_action_path = (
    SCENARIO_DIR / "NEXT_ACTION.md"
)

with open(
    master_json_path,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        V1_SPECIFICATION,
        f,
        indent=2,
        ensure_ascii=False
    )


pd.DataFrame(
    matrix_rows
).to_csv(
    matrix_csv_path,
    index=False
)


pd.DataFrame(
    INPUT_SPEC
).to_csv(
    input_csv_path,
    index=False
)


pd.DataFrame(
    OUTPUT_SPEC
).to_csv(
    output_csv_path,
    index=False
)


validation_report = {

    "notebook":
        "03_V1_DigitalTwin_ScenarioDesign",

    "positive_validation":
        positive_validation,

    "negative_test_count":
        int(len(negative_df)),

    "negative_tests_all_rejected":
        bool(
            negative_df[
                "rejected_as_expected"
            ].all()
        ),

    "v0_lineage_status":
        V0_LINEAGE["status"],

    "scientific_validation_claimed":
        False,

    "whole_pack_localization":
        True,

    "cell37_is_benchmark_only":
        True,

    "random_fault_assignment":
        False,

    "timestamp_utc":
        datetime.datetime.now(
            datetime.timezone.utc
        ).isoformat()
}

validation_report_path.write_text(
    json.dumps(
        validation_report,
        indent=2
    ),
    encoding="utf-8"
)


execution_log = f"""# Notebook 03 Execution Log

Notebook:
03_V1_DigitalTwin_ScenarioDesign

V0 lineage:
{V0_LINEAGE["status"]}

Source backup integrity:
{src_hash == bak_hash}

Positive specification validation:
{positive_validation["valid"]}

Negative tests:
{len(negative_df)} total

Negative tests all rejected:
{bool(negative_df["rejected_as_expected"].all())}

Whole-pack localization:
True

Cell 37 benchmark only:
True

Random fault assignment:
False

Scientific/physical validation claimed:
False

100-cell physical realization:
Not implemented in Notebook 03.

Next notebook:
04_V1_PyBaMM_DataGenerator.ipynb
"""

execution_log_path.write_text(
    execution_log,
    encoding="utf-8"
)


next_action_path.write_text(
    "# NEXT ACTION\n\n"
    "Only proceed after the final Cell 34 gate prints "
    "`GO_AHEAD: NOTEBOOK_03_VERIFIED`.\n\n"
    "Next notebook: "
    "`04_V1_PyBaMM_DataGenerator.ipynb`\n\n"
    "Notebook 03 does not claim physical/scientific validation.\n",
    encoding="utf-8"
)


required_core = [
    master_json_path,
    matrix_csv_path,
    input_csv_path,
    output_csv_path,
    validation_report_path,
    execution_log_path,
    next_action_path,
]

for path in required_core:

    assert (
        path.is_file()
        and path.stat().st_size > 0
    ), (
        f"Missing/empty export: {path}"
    )


print(
    "Cell 28: PASS — "
    "core artifacts exported."
)

Cell 28: PASS — core artifacts exported.


In [48]:
# Cell 29: Full Round-Trip Reload and Content Equality

def _df_canonical(df):
    """
    Canonical representation for content equality checks.

    NaN/None values are normalized to a fixed token and
    all remaining values are compared as strings.
    """
    out = (
        df.copy()
        .fillna("<NA>")
        .astype(str)
        .reset_index(drop=True)
    )

    return out


# ============================================================
# 1. Required core artifacts must exist
# ============================================================

required_roundtrip_artifacts = [
    master_json_path,
    matrix_csv_path,
    input_csv_path,
    output_csv_path,
    validation_report_path,
    execution_log_path,
    next_action_path,
]

for path in required_roundtrip_artifacts:

    assert path.is_file(), (
        f"Required round-trip artifact is missing: {path}"
    )

    assert path.stat().st_size > 0, (
        f"Required round-trip artifact is empty: {path}"
    )


# ============================================================
# 2. Reload master JSON and require exact content equality
# ============================================================

with open(
    master_json_path,
    "r",
    encoding="utf-8"
) as f:

    reloaded_spec = json.load(f)


assert reloaded_spec == V1_SPECIFICATION, (
    "JSON round-trip mismatch against "
    "the in-memory V1_SPECIFICATION."
)


# ============================================================
# 3. Re-run the AUTHORITATIVE validator on reloaded JSON
# ============================================================

reloaded_validation = validate_specification(
    reloaded_spec
)


assert isinstance(
    reloaded_validation,
    dict
), (
    "Reloaded validator result is not a dictionary."
)


assert reloaded_validation.get(
    "valid"
) is True, (
    "Reloaded specification failed authoritative "
    "validation:\n"
    + "\n".join(
        reloaded_validation.get("errors", [])
    )
)


# ============================================================
# 4. Prepare expected in-memory CSV representations
# ============================================================

expected_matrix_df = pd.DataFrame(
    matrix_rows
)

expected_input_df = pd.DataFrame(
    INPUT_SPEC
)

expected_output_df = pd.DataFrame(
    OUTPUT_SPEC
)


# ============================================================
# 5. Reload all three CSV exports
# ============================================================

re_matrix_df = pd.read_csv(
    matrix_csv_path
)

re_input_df = pd.read_csv(
    input_csv_path
)

re_output_df = pd.read_csv(
    output_csv_path
)


# ============================================================
# 6. Exact content equality for Scenario Matrix
# ============================================================

assert _df_canonical(
    re_matrix_df
).equals(
    _df_canonical(
        expected_matrix_df
    )
), (
    "Scenario matrix CSV round-trip "
    "content mismatch."
)


# ============================================================
# 7. Exact content equality for INPUT_SPEC
# ============================================================

assert _df_canonical(
    re_input_df
).equals(
    _df_canonical(
        expected_input_df
    )
), (
    "Input specification CSV round-trip "
    "content mismatch."
)


# ============================================================
# 8. Exact content equality for OUTPUT_SPEC
# ============================================================

assert _df_canonical(
    re_output_df
).equals(
    _df_canonical(
        expected_output_df
    )
), (
    "Output specification CSV round-trip "
    "content mismatch."
)


# ============================================================
# 9. Reload validation_report.json
# ============================================================

reloaded_validation_report = json.loads(
    validation_report_path.read_text(
        encoding="utf-8"
    )
)


assert isinstance(
    reloaded_validation_report,
    dict
), (
    "validation_report.json did not reload "
    "as a JSON object."
)


# Check exact required top-level structure.
required_report_keys = {
    "notebook",
    "positive_validation",
    "negative_test_count",
    "negative_tests_all_rejected",
    "v0_lineage_status",
    "scientific_validation_claimed",
    "timestamp_utc",
}

assert set(
    reloaded_validation_report.keys()
) == required_report_keys, (
    "validation_report.json key set mismatch.\n"
    f"Expected: {sorted(required_report_keys)}\n"
    f"Actual: {sorted(reloaded_validation_report.keys())}"
)


# Check notebook identity.
assert (
    reloaded_validation_report["notebook"]
    == "03_V1_DigitalTwin_ScenarioDesign"
), (
    "Validation report notebook identifier mismatch."
)


# Check positive validation content.
assert (
    reloaded_validation_report[
        "positive_validation"
    ]["valid"] is True
), (
    "validation_report.json does not record "
    "successful positive validation."
)


assert (
    reloaded_validation_report[
        "positive_validation"
    ] == positive_validation
), (
    "validation_report.json positive_validation "
    "content differs from the in-memory result."
)


# Check negative-test count.
assert (
    reloaded_validation_report[
        "negative_test_count"
    ] == len(negative_df)
), (
    "Validation report negative-test count mismatch."
)


# Check that every negative test was rejected.
assert (
    reloaded_validation_report[
        "negative_tests_all_rejected"
    ] is True
), (
    "Validation report indicates that not all "
    "negative tests were rejected."
)


assert (
    bool(
        negative_df[
            "rejected_as_expected"
        ].all()
    )
), (
    "In-memory negative-test suite is not fully passing."
)


# Check V0 lineage status against in-memory authority.
assert (
    reloaded_validation_report[
        "v0_lineage_status"
    ]
    == V0_LINEAGE["status"]
), (
    "Validation report V0 lineage status mismatch."
)


# Scientific validation must remain explicitly false.
assert (
    reloaded_validation_report[
        "scientific_validation_claimed"
    ] is False
), (
    "Notebook 03 must not claim scientific/physical "
    "validation."
)


# Timestamp must exist and be a valid ISO timestamp.
timestamp_text = reloaded_validation_report.get(
    "timestamp_utc"
)

assert isinstance(
    timestamp_text,
    str
) and timestamp_text.strip(), (
    "Validation report timestamp_utc is missing."
)

try:
    datetime.datetime.fromisoformat(
        timestamp_text.replace(
            "Z",
            "+00:00"
        )
    )
except Exception as exc:

    raise AssertionError(
        "validation_report.json timestamp_utc "
        "is not a valid ISO timestamp: "
        f"{timestamp_text}"
    ) from exc


# ============================================================
# 10. Reconstruct expected execution log exactly
# ============================================================

expected_execution_log = f"""# Notebook 03 Execution Log

Notebook:
03_V1_DigitalTwin_ScenarioDesign

V0 lineage:
{V0_LINEAGE["status"]}

Source backup integrity:
{src_hash == bak_hash}

Positive specification validation:
{positive_validation["valid"]}

Negative tests:
{len(negative_df)} total

Negative tests all rejected:
{bool(negative_df["rejected_as_expected"].all())}

Scientific/physical validation claimed:
False

100-cell physical realization:
Not implemented in Notebook 03.

Next notebook:
04_V1_PyBaMM_DataGenerator.ipynb
"""


reloaded_execution_log = (
    execution_log_path.read_text(
        encoding="utf-8"
    )
)


assert (
    reloaded_execution_log
    == expected_execution_log
), (
    "execution_log.md content mismatch "
    "against expected generated content."
)


# ============================================================
# 11. Reconstruct NEXT_ACTION exactly
# ============================================================

expected_next_action = (
    "# NEXT ACTION\n\n"
    "Only proceed after the final Cell 34 gate prints "
    "`GO_AHEAD: NOTEBOOK_03_VERIFIED`.\n\n"
    "Next notebook: "
    "`04_V1_PyBaMM_DataGenerator.ipynb`\n\n"
    "Notebook 03 does not claim physical/scientific validation.\n"
)


reloaded_next_action = (
    next_action_path.read_text(
        encoding="utf-8"
    )
)


assert (
    reloaded_next_action
    == expected_next_action
), (
    "NEXT_ACTION.md content mismatch "
    "against expected generated content."
)


# ============================================================
# 12. Final round-trip status
# ============================================================

ROUNDTRIP_STATUS = "PASS"


print(
    "Cell 29: PASS — complete core artifact "
    "round-trip verified."
)

print(
    "  ✓ Master JSON exact equality"
)

print(
    "  ✓ Reloaded JSON revalidated"
)

print(
    "  ✓ Scenario matrix CSV exact equality"
)

print(
    "  ✓ Input specification CSV exact equality"
)

print(
    "  ✓ Output specification CSV exact equality"
)

print(
    "  ✓ Validation report content verified"
)

print(
    "  ✓ Execution log content verified"
)

print(
    "  ✓ NEXT_ACTION content verified"
)

print(
    f"  ✓ Negative tests preserved: "
    f"{len(negative_df)}/{len(negative_cases)} rejected"
)

print(
    "  ✓ Scientific validation remains explicitly FALSE"
)

Cell 29: PASS — full JSON equality and core CSV content equality verified after reload.


In [49]:
# Cell 30: Requirements Status Matrix

REQUIREMENTS_COLUMNS = [
    "requirement", "specification_status", "implementation_status",
    "validation_status", "evidence", "blocker", "next_stage", "remaining_work"
]

requirements_rows = [
    {
        "requirement": "Genuine V0 lineage", "specification_status": "VERIFIED", "implementation_status": "AVAILABLE",
        "validation_status": "VERIFIED_FROM_ARTIFACTS", "evidence": "Exact 43200x16 schema, metadata linkage, 20 conditions, 12/6/2 split and hashes.",
        "blocker": None, "next_stage": "04_V1_PyBaMM_DataGenerator.ipynb", "remaining_work": "Maintain V0 lineage."
    },
    {
        "requirement": "100-cell whole-pack monitoring", "specification_status": "VERIFIED", "implementation_status": "DESIGN_ONLY",
        "validation_status": "STRUCTURAL_ONLY", "evidence": "Cell IDs 1..100 and whole-pack localization contract.",
        "blocker": "Physical pack implementation is downstream.", "next_stage": "04_V1_PyBaMM_DataGenerator.ipynb", "remaining_work": "Generate cell-level physical states for all 100 cells."
    },
    {
        "requirement": "Cell 37 benchmark experiment", "specification_status": "VERIFIED", "implementation_status": "DESIGN_ONLY",
        "validation_status": "STRUCTURAL_ONLY", "evidence": "Known-location benchmark, not permanent fault location.",
        "blocker": "Physical degradation mapping unresolved.", "next_stage": "04_V1_PyBaMM_DataGenerator.ipynb", "remaining_work": "Generate matched healthy/degraded benchmark cases."
    },
    {
        "requirement": "Dynamic defect localization across Cells 1-100", "specification_status": "VERIFIED", "implementation_status": "DESIGN_ONLY",
        "validation_status": "NOT_YET_PHYSICAL", "evidence": "Per-cell outputs and all-cell candidate scan required.",
        "blocker": "Requires cell-level physical simulation/data and validated localization logic.", "next_stage": "04+ downstream", "remaining_work": "Evaluate every cell and classify defective/affected/potentially affected/healthy."
    },
    {
        "requirement": "Complete V1 INPUT_SPEC", "specification_status": "VERIFIED", "implementation_status": "SCHEMA_ONLY",
        "validation_status": "STRUCTURAL", "evidence": "Required fields/classification metadata validated.",
        "blocker": None, "next_stage": "04_V1_PyBaMM_DataGenerator.ipynb", "remaining_work": "Populate real downstream data."
    },
    {
        "requirement": "Complete V1 OUTPUT_SPEC", "specification_status": "VERIFIED", "implementation_status": "SCHEMA_ONLY",
        "validation_status": "STRUCTURAL", "evidence": "Physics outputs plus per-cell localization outputs are defined.",
        "blocker": "Physical implementation downstream.", "next_stage": "04_V1_PyBaMM_DataGenerator.ipynb", "remaining_work": "Generate and validate trajectories for all cells."
    },
    {
        "requirement": "S1-S6 whole-pack scenario suite", "specification_status": "VERIFIED", "implementation_status": "DESIGN_ONLY",
        "validation_status": "STRUCTURAL", "evidence": "S1-S6 and ten scenario rows; each case scans 100 cells.",
        "blocker": None, "next_stage": "04_V1_PyBaMM_DataGenerator.ipynb", "remaining_work": "Execute scenarios physically."
    },
    {
        "requirement": "S6 counterfactual intervention", "specification_status": "VERIFIED", "implementation_status": "DESIGN_ONLY",
        "validation_status": "STRUCTURAL", "evidence": "Five C-rate candidates, no winner, all-cell rescan and S5-state invariance.",
        "blocker": "Validated constraints unavailable yet.", "next_stage": "Downstream evaluation", "remaining_work": "Run full-horizon candidate simulations."
    },
    {
        "requirement": "B0-B3 comparison", "specification_status": "VERIFIED", "implementation_status": "DESIGN_ONLY",
        "validation_status": "STRUCTURAL", "evidence": "Canonical B0/B1/B2/B3 mapping.",
        "blocker": "Comparison experiments are downstream.", "next_stage": "Model/evaluation notebooks", "remaining_work": "Run controlled comparisons."
    },
    {
        "requirement": "Risk engine", "specification_status": "VERIFIED", "implementation_status": "SPECIFICATION_ONLY",
        "validation_status": "UNRESOLVED", "evidence": "Six domains and threshold provenance schema.",
        "blocker": "Operational boundaries unresolved.", "next_stage": "Physical validation", "remaining_work": "Source and validate chemistry/model-specific limits."
    },
    {
        "requirement": "Early warning", "specification_status": "VERIFIED", "implementation_status": "SPECIFICATION_ONLY",
        "validation_status": "NOT_YET_EVALUATED", "evidence": "Alarm semantics and metrics defined.",
        "blocker": "Requires ground-truth events and validated thresholds.", "next_stage": "Downstream evaluation", "remaining_work": "Measure lead time, false/missed events, timing error and latency."
    },
    {
        "requirement": "No fabricated results", "specification_status": "VERIFIED", "implementation_status": "ENFORCED",
        "validation_status": "PASS", "evidence": "Notebook 03 contains specification only; fabricated result containers are forbidden.",
        "blocker": None, "next_stage": "All downstream notebooks", "remaining_work": "Maintain evidence discipline."
    }
]

reqs_df = pd.DataFrame(requirements_rows, columns=REQUIREMENTS_COLUMNS)
reqs_csv_path = SCENARIO_DIR / "requirements_status.csv"
reqs_df.to_csv(reqs_csv_path, index=False)

assert reqs_df.columns.tolist() == REQUIREMENTS_COLUMNS
assert reqs_csv_path.is_file() and reqs_csv_path.stat().st_size > 0

print(f"Cell 30: PASS — requirements matrix exported with {len(reqs_df)} tracked requirements.")
display(reqs_df)


Cell 30: PASS — requirements matrix exported with 12 tracked requirements.


,requirement,specification_status,implementation_status,validation_status,evidence,blocker,next_stage,remaining_work
0,Genuine V0 lineage,VERIFIED,AVAILABLE,VERIFIED_FROM_ARTIFACTS,"Exact 43200x16 schema, metadata linkage, 20 co...",None,04_V1_PyBaMM_DataGenerator.ipynb,Maintain V0 lineage.
1,100-cell whole-pack monitoring,VERIFIED,DESIGN_ONLY,STRUCTURAL_ONLY,Cell IDs 1..100 and whole-pack localization co...,Physical pack implementation is downstream.,04_V1_PyBaMM_DataGenerator.ipynb,Generate cell-level physical states for all 10...
2,Cell 37 benchmark experiment,VERIFIED,DESIGN_ONLY,STRUCTURAL_ONLY,"Known-location benchmark, not permanent fault ...",Physical degradation mapping unresolved.,04_V1_PyBaMM_DataGenerator.ipynb,Generate matched healthy/degraded benchmark ca...
3,Dynamic defect localization across Cells 1-100,VERIFIED,DESIGN_ONLY,NOT_YET_PHYSICAL,Per-cell outputs and all-cell candidate scan r...,Requires cell-level physical simulation/data a...,04+ downstream,Evaluate every cell and classify defective/aff...
4,Complete V1 INPUT_SPEC,VERIFIED,SCHEMA_ONLY,STRUCTURAL,Required fields/classification metadata valida...,None,04_V1_PyBaMM_DataGenerator.ipynb,Populate real downstream data.
5,Complete V1 OUTPUT_SPEC,VERIFIED,SCHEMA_ONLY,STRUCTURAL,Physics outputs plus per-cell localization out...,Physical implementation downstream.,04_V1_PyBaMM_DataGenerator.ipynb,Generate and validate trajectories for all cells.
6,S1-S6 whole-pack scenario suite,VERIFIED,DESIGN_ONLY,STRUCTURAL,S1-S6 and ten scenario rows; each case scans 1...,None,04_V1_PyBaMM_DataGenerator.ipynb,Execute scenarios physically.
7,S6 counterfactual intervention,VERIFIED,DESIGN_ONLY,STRUCTURAL,"Five C-rate candidates, no winner, all-cell re...",Validated constraints unavailable yet.,Downstream evaluation,Run full-horizon candidate simulations.
8,B0-B3 comparison,VERIFIED,DESIGN_ONLY,STRUCTURAL,Canonical B0/B1/B2/B3 mapping.,Comparison experiments are downstream.,Model/evaluation notebooks,Run controlled comparisons.
9,Risk engine,VERIFIED,SPECIFICATION_ONLY,UNRESOLVED,Six domains and threshold provenance schema.,Operational boundaries unresolved.,Physical validation,Source and validate chemistry/model-specific l...


In [50]:

# Cell 31: Continuation State & Downstream Handoff

continuation_state = {
    "current_stage":
        "03_V1_DigitalTwin_ScenarioDesign",

    "status":
        "NOTEBOOK_03_SPECIFICATION_COMPLETE_PENDING_FINAL_GATE",

    "scientific_validation_claimed":
        False,

    "next_notebook":
        "04_V1_PyBaMM_DataGenerator.ipynb",

    "forbidden_downstream_name":
        "04_V1_Physical_Simulator",

    "handoff_artifacts": {
        "specification_json": str(master_json_path),
        "scenario_matrix": str(matrix_csv_path),
        "input_spec": str(input_csv_path),
        "output_spec": str(output_csv_path),
        "requirements_status": str(reqs_csv_path),
        "validation_report": str(validation_report_path),
        "execution_log": str(execution_log_path),
        "next_action": str(next_action_path),
    },
}

continuation_path = (
    SCENARIO_DIR / "continuation_state.json"
)

continuation_path.write_text(
    json.dumps(
        continuation_state,
        indent=2
    ),
    encoding="utf-8"
)

assert continuation_path.is_file()
assert continuation_path.stat().st_size > 0

print(
    "Cell 31: PASS — continuation state and exact downstream handoff recorded."
)


Cell 31: PASS — continuation state and exact downstream handoff recorded.


In [51]:

# Cell 32: Cryptographic SHA-256 Manifest

authoritative_files = [
    Path(source_nb),
    Path(BACKUP_PATH),
    Path(master_json_path),
    Path(matrix_csv_path),
    Path(input_csv_path),
    Path(output_csv_path),
    Path(reqs_csv_path),
    Path(validation_report_path),
    Path(execution_log_path),
    Path(next_action_path),
    Path(continuation_path),
]

manifest_entries = {}

for path in authoritative_files:
    assert path.is_file(), (
        f"Authoritative artifact missing: {path}"
    )

    data = path.read_bytes()

    manifest_entries[path.name] = {
        "path": str(path),
        "size_bytes": len(data),
        "sha256": hashlib.sha256(data).hexdigest(),
    }

manifest_path = (
    SCENARIO_DIR / "artifact_hash_manifest.json"
)

manifest_obj = {
    "manifest_version": "1.0",
    "scope": "Notebook 03 authoritative artifacts",
    "artifacts": manifest_entries,
    "self_hash_note":
        "The manifest is not self-referenced to avoid recursive hashing.",
}

manifest_path.write_text(
    json.dumps(
        manifest_obj,
        indent=2
    ),
    encoding="utf-8"
)

assert manifest_path.is_file()
assert manifest_path.stat().st_size > 0

print(
    "Cell 32: PASS — SHA-256 manifest written for "
    f"{len(manifest_entries)} authoritative artifacts."
)


Cell 32: PASS — SHA-256 manifest written for 11 authoritative artifacts.


In [52]:
# Cell 33: Evidence-Driven Self-Audit

required_exports = [
    master_json_path,
    matrix_csv_path,
    input_csv_path,
    output_csv_path,
    reqs_csv_path,
    validation_report_path,
    continuation_path,
    execution_log_path,
    next_action_path,
    manifest_path,
]

audit_rows = [
    ("Genuine V0 lineage", V0_LINEAGE["status"] == "VERIFIED"),
    ("V0 exact schema", v0_df.shape == (43200, 16) and v0_df.columns.tolist() == EXPECTED_V0_COLUMNS),
    ("V0 condition/split structure", V0_LINEAGE["condition_count"] == 20 and V0_LINEAGE["split_condition_counts"] == {"train": 12, "val": 6, "test": 2}),
    ("Dataset/metadata condition linkage", V0_LINEAGE["dataset_condition_ids"] == V0_LINEAGE["metadata_condition_ids"]),
    ("Source backup integrity", src_hash == bak_hash and len(src_bytes) == len(bak_bytes)),
    ("100 unique cell IDs", PACK_CONFIG["monitored_cells"] == list(range(1, 101))),
    ("Whole-pack localization", PACK_CONFIG["fault_localization"]["candidate_cells"] == list(range(1, 101))),
    ("No random fault assignment", PACK_CONFIG["fault_localization"]["random_fault_assignment"] is False),
    ("Cell 37 benchmark only", PACK_CONFIG["reference_experiment_cell_id"] == 37 and V1_SPECIFICATION["boundary_declarations"]["cell37_is_benchmark_not_permanent_fault"] is True),
    ("Whole-pack output coverage", all(f in {r.get("field") for r in OUTPUT_SPEC} for f in ["per_cell_health_state", "per_cell_anomaly_score", "defective_cells", "affected_cells", "potentially_affected_cells", "healthy_cells", "localization_evidence"])),
    ("Scenario matrix scans all cells", len(scenario_matrix_df) == 10 and scenario_matrix_df["cells_scanned"].eq(100).all()),
    ("S6 candidates", sorted(S6_INTERVENTION_SWEEP["tested_candidate_c_rates"]) == [0.4, 0.8, 1.0, 1.5, 2.0]),
    ("S6 no preselected winner", S6_INTERVENTION_SWEEP["winner_preselected"] is None),
    ("S6 inherits affected cells", S6_INTERVENTION_SWEEP["affected_cells_source"] == "INHERIT_FROM_S5"),
    ("B0-B3 mapping", [row["model_id"] for row in BASELINE_SPEC] == ["B0", "B1", "B2", "B3"]),
    ("Six unresolved risk domains", len(RISK_SPEC["domains"]) == 6 and RISK_SPEC["operational_status"] == "UNASSESSED_UNTIL_VALIDATED"),
    ("Early-warning contract", len(EARLY_WARNING_SPEC["metrics"]) >= 6),
    ("Positive validation", positive_validation["valid"]),
    ("Negative tests all rejected", bool(negative_df["rejected_as_expected"].all())),
    ("Round-trip", ROUNDTRIP_STATUS == "PASS"),
    ("All required exports present", all(path.is_file() and path.stat().st_size > 0 for path in required_exports)),
    ("Requirements matrix schema", reqs_df.columns.tolist() == REQUIREMENTS_COLUMNS),
    ("No fabricated results", all(key not in V1_SPECIFICATION for key in ["results", "predictions", "evaluation_results", "trained_model_results", "fabricated_results"])),
    ("Correct downstream handoff", continuation_state["next_notebook"] == "04_V1_PyBaMM_DataGenerator.ipynb"),
]

audit_table = pd.DataFrame(audit_rows, columns=["audit_gate", "passed"])
audit_failures = audit_table.loc[~audit_table["passed"], "audit_gate"].tolist()
SELF_AUDIT_STATUS = "PASS" if not audit_failures else "FAIL"

display(audit_table)
if SELF_AUDIT_STATUS == "PASS":
    print(f"Cell 33: PASS — all {len(audit_table)} self-audit gates passed.")
else:
    print("Cell 33: FAIL — blockers:", audit_failures)


,audit_gate,passed
0,Genuine V0 lineage,True
1,V0 exact schema,True
2,V0 condition/split structure,True
3,Dataset/metadata condition linkage,True
4,Source backup integrity,True
5,100 unique cell IDs,True
6,Whole-pack localization,True
7,No random fault assignment,True
8,Cell 37 benchmark only,True
9,Whole-pack output coverage,True


Cell 33: PASS — all 24 self-audit gates passed.


In [53]:

# Cell 33: Final Evidence Recheck Before Decision Gate

required_exports = [
    master_json_path,
    matrix_csv_path,
    input_csv_path,
    output_csv_path,
    reqs_csv_path,
    validation_report_path,
    continuation_path,
    execution_log_path,
    next_action_path,
    manifest_path,
]

recheck_rows = []

recheck_rows.append((
    "All required exports exist",
    all(
        p.is_file() and p.stat().st_size > 0
        for p in required_exports
    )
))

reloaded_reqs_df = pd.read_csv(reqs_csv_path)
recheck_rows.append((
    "Requirements CSV round-trip",
    _df_canonical(reloaded_reqs_df).equals(
        _df_canonical(reqs_df)
    )
))

reloaded_continuation = json.loads(
    continuation_path.read_text(encoding="utf-8")
)
recheck_rows.append((
    "Continuation JSON round-trip",
    reloaded_continuation == continuation_state
))

reloaded_validation_report = json.loads(
    validation_report_path.read_text(
        encoding="utf-8"
    )
)
recheck_rows.append((
    "Validation report consistency",
    reloaded_validation_report["v0_lineage_status"]
    == "VERIFIED"
    and reloaded_validation_report[
        "scientific_validation_claimed"
    ] is False
))

reloaded_next_action = (
    next_action_path.read_text(
        encoding="utf-8"
    )
)
recheck_rows.append((
    "Next-action handoff",
    "04_V1_PyBaMM_DataGenerator.ipynb"
    in reloaded_next_action
))

manifest_obj = json.loads(
    manifest_path.read_text(encoding="utf-8")
)

manifest_ok = True

for name, record in manifest_obj.get(
    "artifacts", {}
).items():

    path = Path(record["path"])

    if not path.is_file():
        manifest_ok = False
        break

    data = path.read_bytes()

    if len(data) != record["size_bytes"]:
        manifest_ok = False
        break

    if hashlib.sha256(
        data
    ).hexdigest() != record["sha256"]:
        manifest_ok = False
        break

recheck_rows.append((
    "Manifest integrity",
    manifest_ok
))

recheck_rows.append((
    "Scientific validation not claimed",
    V1_SPECIFICATION[
        "boundary_declarations"
    ]["no_fabricated_predictions"] is True
    and V1_SPECIFICATION[
        "boundary_declarations"
    ]["no_training_in_notebook_03"] is True
))

final_evidence_table = pd.DataFrame(
    recheck_rows,
    columns=["gate", "passed"]
)

FINAL_EVIDENCE_STATUS = (
    "PASS"
    if bool(final_evidence_table["passed"].all())
    else "FAIL"
)

display(final_evidence_table)

if FINAL_EVIDENCE_STATUS == "PASS":
    print(
        "Cell 33: PASS — final evidence recheck passed."
    )
else:
    print(
        "Cell 33: FAIL — final evidence recheck failed."
    )


,gate,passed
0,All required exports exist,True
1,Requirements CSV round-trip,True
2,Continuation JSON round-trip,True
3,Validation report consistency,True
4,Next-action handoff,True
5,Manifest integrity,True
6,Scientific validation not claimed,True


Cell 33: PASS — final evidence recheck passed.


In [54]:
# Cell 34: Conditional Final GO / NO-GO Decision Gate

mandatory_gates = {
    "v0_lineage": V0_LINEAGE["status"] == "VERIFIED",
    "source_backup": src_hash == bak_hash and len(src_bytes) == len(bak_bytes),
    "positive_validation": positive_validation["valid"],
    "negative_tests": bool(negative_df["rejected_as_expected"].all()),
    "round_trip": ROUNDTRIP_STATUS == "PASS",
    "required_exports": all(path.is_file() and path.stat().st_size > 0 for path in required_exports),
    "requirements_schema": reqs_df.columns.tolist() == REQUIREMENTS_COLUMNS,
    "self_audit": SELF_AUDIT_STATUS == "PASS",
    "final_evidence_recheck": FINAL_EVIDENCE_STATUS == "PASS",
    "correct_handoff": continuation_state["next_notebook"] == "04_V1_PyBaMM_DataGenerator.ipynb",
    "whole_pack_localization": PACK_CONFIG["fault_localization"]["mode"] == "DYNAMIC_WHOLE_PACK" and PACK_CONFIG["fault_localization"]["candidate_cells"] == list(range(1, 101)),
    "cell37_benchmark_only": V1_SPECIFICATION["boundary_declarations"]["cell37_is_benchmark_not_permanent_fault"] is True,
    "no_random_fault_assignment": V1_SPECIFICATION["boundary_declarations"]["random_fault_assignment_for_final_results_forbidden"] is True,
    "no_fabricated_results": all(key not in V1_SPECIFICATION for key in ["results", "predictions", "evaluation_results", "trained_model_results", "fabricated_results"]),
}

all_mandatory = all(mandatory_gates.values())

print("=" * 80)
print("FINAL DECISION GATE: 03_V1_DigitalTwin_ScenarioDesign")
print("=" * 80)
for gate, passed in mandatory_gates.items():
    print(f"{gate:30s}: {'PASS' if passed else 'FAIL'}")
print()

if all_mandatory:
    print("GO_AHEAD: NOTEBOOK_03_VERIFIED")
    print("Whole-pack dynamic localization is specified but not scientifically validated here.")
    print("Cell 37 is only the controlled benchmark experiment, not a permanent fault location.")
    print("Next notebook: 04_V1_PyBaMM_DataGenerator.ipynb")
else:
    print("DO_NOT_PROCEED")
    print("One or more mandatory gates failed. Review the failed gates above.")

print("=" * 80)


FINAL DECISION GATE: 03_V1_DigitalTwin_ScenarioDesign
v0_lineage                    : PASS
source_backup                 : PASS
positive_validation           : PASS
negative_tests                : PASS
round_trip                    : PASS
required_exports              : PASS
requirements_schema           : PASS
self_audit                    : PASS
final_evidence_recheck        : PASS
correct_handoff               : PASS
whole_pack_localization       : PASS
cell37_benchmark_only         : PASS
no_random_fault_assignment    : PASS
no_fabricated_results         : PASS

GO_AHEAD: NOTEBOOK_03_VERIFIED
Whole-pack dynamic localization is specified but not scientifically validated here.
Cell 37 is only the controlled benchmark experiment, not a permanent fault location.
Next notebook: 04_V1_PyBaMM_DataGenerator.ipynb
